### **Installations, Configurations, Imports, Setups and Data Preparation**

In [3]:
import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = {
    "torch": "torch",
    "transformers": "transformers>=4.51,<5",
    "datasets": "datasets",
    "pandas": "pandas",
    "numpy": "numpy",
    "sacrebleu": "sacrebleu>=2.4,<3",
    "sentencepiece": "sentencepiece",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm"
}

missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All dependencies are installed.")

print("Python:", sys.version)
print("Executable:", sys.executable)

All dependencies are installed.
Python: 3.11.15 (main, Jun 11 2026, 15:20:16) [GCC 14.3.0]
Executable: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/bin/python


In [4]:
import os
import sys
import gc
import re
import ast
import json
import math
import time
import random
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import sacrebleu
import torch
import torch.nn as nn
import torch.nn.functional as F

from IPython.display import display
from sacrebleu.metrics import BLEU, CHRF
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.set_float32_matmul_precision("high")

In [5]:
# ============================================================
# EDIT PATHS AND TRAINING PARAMETERS ONLY IN THIS CELL
# Change RUN_NAME whenever starting a materially different run.
# ============================================================

PROJECT_DIR = Path(os.environ.get("AXMT_HOME", str(Path.home() / "alexandriax_mt_14d"))).expanduser()
INFERENCE_VARIANTS_ROOT = PROJECT_DIR / "inference_variants"
SHARED_CACHE_DIR = INFERENCE_VARIANTS_ROOT / "_shared_cache"
PREPARED_CACHE_DIR = SHARED_CACHE_DIR / "paired_dev_train_v3"

RUN_NAME = "94_xlmr_large_dialect_crossencoder_pairrank_v1"
RUN_DIR = PROJECT_DIR / "rerankers" / RUN_NAME
OUTPUT_DIR = INFERENCE_VARIANTS_ROOT / RUN_NAME

LOCAL_ENCODER_DIR = PROJECT_DIR / "models" / "hf" / "xlm-roberta-large"
ENCODER_NAME = str(LOCAL_ENCODER_DIR) if LOCAL_ENCODER_DIR.exists() else "FacebookAI/xlm-roberta-large"

CANDIDATE_VARIANTS = [
    "00_previous_official_control",
    "01_exact_training_parity",
    "02_metadata_no_shots",
    "03_retrieved_two_shot",
    "04_training_parity_with_participants",
    "05_retrieved_two_shot_with_participants",
    "06_ckpt16500_retrieved_two_shot",
    "07_ckpt16000_retrieved_two_shot",
    "08_interp_015_035_050_retrieved_two_shot"
]

BASELINE_VARIANT = "92_mixed_best_checkpoint_variant_per_country"

EXPECTED_DEV_TURNS = 12250
EXPECTED_TRAIN_TURNS = 66480
EXPECTED_DEV_COUNTRIES = 11

# ------------------------------------------------------------
# DIALECT-RESOURCE TRAINING PARAMETERS
# ------------------------------------------------------------

DIALECT_MAX_LENGTH = 160
DIALECT_BATCH_SIZE = 16
DIALECT_GRAD_ACCUM_STEPS = 2
DIALECT_EPOCHS = 2
DIALECT_LR = 1e-5
DIALECT_SAVE_STEPS = 250

# ------------------------------------------------------------
# PAIRWISE RERANKER TRAINING PARAMETERS
# ------------------------------------------------------------

N_FOLDS = 5
RANK_MAX_LENGTH = 384
RANK_BATCH_SIZE = 4
RANK_GRAD_ACCUM_STEPS = 8
RANK_EPOCHS = 2
RANK_ENCODER_LR = 1e-5
RANK_HEAD_LR = 5e-5
RANK_SAVE_STEPS = 250
RANK_TEMPERATURE = 1.0

PAIRS_PER_TURN = 6
MIN_UTILITY_GAP = 0.05
PAIR_WEIGHT_MIN = 0.25
PAIR_WEIGHT_MAX = 4.0

LABEL_SPBLEU_WEIGHT = 0.80
LABEL_CHRF_WEIGHT = 0.20

# ------------------------------------------------------------
# SHARED OPTIMIZATION AND INFERENCE PARAMETERS
# ------------------------------------------------------------

WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

SCORE_BATCH_SIZE = 32
SCORE_SAVE_ROWS = 128
NUM_WORKERS = 0

# 0.0 means only exact score ties fall back to System92.
MIN_SCORE_MARGIN = 0.0

# Package the reranker only when its OOF spBLEU beats System92.
DEPLOY_MIN_SPBLEU_GAIN = 0.0

DIALECT_NAMES = {
    "EG": "Egyptian Arabic",
    "JO": "Jordanian Arabic",
    "LB": "Lebanese Arabic",
    "MA": "Moroccan Arabic",
    "MR": "Mauritanian Arabic",
    "OM": "Omani Arabic",
    "PS": "Palestinian Arabic",
    "SA": "Saudi Arabic",
    "SY": "Syrian Arabic",
    "TN": "Tunisian Arabic",
    "YE": "Yemeni Arabic"
}

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for XLM-R-large training.")

USE_BF16 = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
DEVICE = torch.device("cuda")

RUN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def short_hash(value):
    payload = json.dumps(value, sort_keys=True).encode()
    return hashlib.sha256(payload).hexdigest()[:16]

DIALECT_SIGNATURE = short_hash({
    "encoder": ENCODER_NAME,
    "max_length": DIALECT_MAX_LENGTH,
    "batch": DIALECT_BATCH_SIZE,
    "accum": DIALECT_GRAD_ACCUM_STEPS,
    "epochs": DIALECT_EPOCHS,
    "lr": DIALECT_LR,
    "seed": SEED
})

RANK_SIGNATURE = short_hash({
    "dialect": DIALECT_SIGNATURE,
    "variants": CANDIDATE_VARIANTS,
    "folds": N_FOLDS,
    "max_length": RANK_MAX_LENGTH,
    "batch": RANK_BATCH_SIZE,
    "accum": RANK_GRAD_ACCUM_STEPS,
    "epochs": RANK_EPOCHS,
    "encoder_lr": RANK_ENCODER_LR,
    "head_lr": RANK_HEAD_LR,
    "pairs": PAIRS_PER_TURN,
    "min_gap": MIN_UTILITY_GAP,
    "label_weights": [LABEL_SPBLEU_WEIGHT, LABEL_CHRF_WEIGHT],
    "seed": SEED
})

print("Run:", RUN_NAME)
print("Encoder:", ENCODER_NAME)
print("GPU:", torch.cuda.get_device_name(0))
print("Training dtype:", AMP_DTYPE)
print("Run directory:", RUN_DIR)
print("Output directory:", OUTPUT_DIR)

Run: 94_xlmr_large_dialect_crossencoder_pairrank_v1
Encoder: FacebookAI/xlm-roberta-large
GPU: NVIDIA GeForce RTX 5090
Training dtype: torch.bfloat16
Run directory: /home/mabdallah/alexandriax_mt_14d/rerankers/94_xlmr_large_dialect_crossencoder_pairrank_v1
Output directory: /home/mabdallah/alexandriax_mt_14d/inference_variants/94_xlmr_large_dialect_crossencoder_pairrank_v1


In [6]:
DEV_CACHE_PATH = PREPARED_CACHE_DIR / "official_dev_df.pkl"
TRAIN_CACHE_PATH = PREPARED_CACHE_DIR / "train_fewshot_pool_df.pkl"

if not DEV_CACHE_PATH.exists() or not TRAIN_CACHE_PATH.exists():
    raise FileNotFoundError(
        f"Prepared cache missing under {PREPARED_CACHE_DIR}. "
        "Run the data-preparation cell in Inference_Variants first."
    )

official_dev_df = pd.read_pickle(DEV_CACHE_PATH)
official_dev_df = official_dev_df.sort_values(
    ["config", "conversation_id", "turn_order"]
).reset_index(drop=True)

train_gold_df = pd.read_pickle(TRAIN_CACHE_PATH).reset_index(drop=True)

for column in ["source_id", "config", "conversation_id", "source_text", "reference_arabic"]:
    official_dev_df[column] = official_dev_df[column].fillna("").astype(str)

for column in ["config", "conversation_id", "target_arabic"]:
    train_gold_df[column] = train_gold_df[column].fillna("").astype(str)

official_dev_df["turn_order"] = pd.to_numeric(
    official_dev_df["turn_order"], errors="raise"
).astype(int)

if len(official_dev_df) != EXPECTED_DEV_TURNS:
    raise RuntimeError(f"Expected {EXPECTED_DEV_TURNS} DEV turns, found {len(official_dev_df)}.")

if len(train_gold_df) != EXPECTED_TRAIN_TURNS:
    raise RuntimeError(f"Expected {EXPECTED_TRAIN_TURNS} train turns, found {len(train_gold_df)}.")

if official_dev_df["config"].nunique() != EXPECTED_DEV_COUNTRIES:
    raise RuntimeError("Unexpected number of DEV countries.")

def locate_prediction_csv(folder):
    for filename in ["turn_predictions.csv", "scored_turn_predictions.csv"]:
        path = folder / filename
        if path.exists():
            return path
    raise FileNotFoundError(f"No turn prediction CSV found under {folder}")

def load_ordered_prediction(folder, keep_selected=False):
    raw = pd.read_csv(locate_prediction_csv(folder))

    if "source_id" not in raw or "prediction" not in raw:
        raise ValueError(f"Bad prediction schema: {folder}")

    raw["source_id"] = raw["source_id"].astype(str)
    raw["prediction"] = raw["prediction"].fillna("").astype(str).str.strip()

    if raw["source_id"].duplicated().any() or (raw["prediction"] == "").any():
        raise RuntimeError(f"Duplicate or empty prediction in {folder}")

    columns = ["source_id", "prediction"]

    if keep_selected and "selected_from_variant" in raw:
        columns.append("selected_from_variant")

    ordered = official_dev_df[["source_id"]].merge(
        raw[columns], on="source_id", how="left", validate="one_to_one"
    )

    if ordered["prediction"].isna().any():
        raise RuntimeError(f"Incomplete predictions in {folder}")

    return ordered

variant_names = list(CANDIDATE_VARIANTS)

candidate_frames = [
    load_ordered_prediction(INFERENCE_VARIANTS_ROOT / name)
    for name in variant_names
]

candidate_texts = np.column_stack([
    frame["prediction"].to_numpy(dtype=object)
    for frame in candidate_frames
])

baseline_frame = load_ordered_prediction(
    INFERENCE_VARIANTS_ROOT / BASELINE_VARIANT,
    keep_selected=True
)

baseline_predictions = baseline_frame["prediction"].to_numpy(dtype=object)

def normalized_text(value):
    return re.sub(r"\s+", " ", str(value)).strip()

variant_to_idx = {name: index for index, name in enumerate(variant_names)}
baseline_idx = np.full(len(official_dev_df), -1, dtype=np.int16)

for row_index, baseline_text in enumerate(baseline_predictions):
    selected_name = ""

    if "selected_from_variant" in baseline_frame:
        selected_name = str(
            baseline_frame.loc[row_index, "selected_from_variant"]
        ).strip()

    candidate_index = variant_to_idx.get(selected_name, -1)

    if (
        candidate_index < 0
        or normalized_text(candidate_texts[row_index, candidate_index])
        != normalized_text(baseline_text)
    ):
        matches = [
            index
            for index in range(len(variant_names))
            if normalized_text(candidate_texts[row_index, index])
            == normalized_text(baseline_text)
        ]

        candidate_index = matches[0] if matches else -1

    if candidate_index < 0:
        source_id = official_dev_df.loc[row_index, "source_id"]
        raise RuntimeError(
            f"System92 prediction is absent from the candidate pool: {source_id}"
        )

    baseline_idx[row_index] = candidate_index

candidate_hasher = hashlib.sha256()

for name in variant_names:
    candidate_hasher.update(name.encode())

for text in candidate_texts.reshape(-1):
    candidate_hasher.update(str(text).encode("utf-8"))
    candidate_hasher.update(b"\0")

CANDIDATE_HASH = candidate_hasher.hexdigest()[:20]
N, K = candidate_texts.shape

print("DEV rows:", len(official_dev_df))
print("Gold training rows:", len(train_gold_df))
print("Candidates per turn:", K)
print("Candidate variants:", variant_names)
print("Candidate hash:", CANDIDATE_HASH)

DEV rows: 12250
Gold training rows: 66480
Candidates per turn: 9
Candidate variants: ['00_previous_official_control', '01_exact_training_parity', '02_metadata_no_shots', '03_retrieved_two_shot', '04_training_parity_with_participants', '05_retrieved_two_shot_with_participants', '06_ckpt16500_retrieved_two_shot', '07_ckpt16000_retrieved_two_shot', '08_interp_015_035_050_retrieved_two_shot']
Candidate hash: 4442f3c1d2e6d6dfaeb3


### **Create conversation-grouped OOF folds**

In [5]:
groups = (
    official_dev_df["config"]
    + "::"
    + official_dev_df["conversation_id"]
)

fold_id = np.full(N, -1, dtype=np.int8)

splitter = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

for fold, (_, validation_indices) in enumerate(
    splitter.split(
        np.zeros(N),
        official_dev_df["config"],
        groups
    )
):
    fold_id[validation_indices] = fold

if (fold_id < 0).any():
    raise RuntimeError("Fold assignment is incomplete.")

fold_check = (
    official_dev_df
    .assign(fold=fold_id)
    .groupby(["config", "conversation_id"])["fold"]
    .nunique()
    .max()
)

if fold_check != 1:
    raise RuntimeError("Conversation leakage across folds.")

display(pd.crosstab(
    official_dev_df["config"],
    fold_id,
    margins=True
))

col_0,0,1,2,3,4,All
config,,,,,,
EG,224,223,222,222,222,1113
JO,223,224,222,222,222,1113
LB,223,223,223,225,224,1118
MA,223,221,222,222,222,1110
MR,222,222,224,224,222,1114
OM,222,221,221,222,223,1109
PS,222,223,221,221,223,1110
SA,222,221,221,223,223,1110
SY,223,225,225,223,223,1119


### **Define the models**

In [6]:
def configure_encoder(encoder):
    encoder.config.use_cache = False

    if hasattr(encoder, "gradient_checkpointing_enable"):
        encoder.gradient_checkpointing_enable()

    return encoder

class DialectClassifier(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()

        self.encoder = configure_encoder(
            AutoModel.from_pretrained(model_name)
        )

        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.10)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        hidden = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0]

        return self.classifier(self.dropout(hidden))

class DialectAwareQualityModel(nn.Module):
    def __init__(self, model_name, dialect_feature_size=3):
        super().__init__()

        self.encoder = configure_encoder(
            AutoModel.from_pretrained(model_name)
        )

        hidden_size = self.encoder.config.hidden_size

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size + dialect_feature_size),
            nn.Linear(hidden_size + dialect_feature_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, input_ids, attention_mask, dialect_features):
        hidden = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0]

        combined = torch.cat([hidden, dialect_features], dim=-1)
        return self.head(combined).squeeze(-1)

### **Checkpoint and mixed-precision helpers**

In [7]:
def amp_context():
    return torch.autocast("cuda", dtype=AMP_DTYPE)

def new_scaler():
    try:
        return torch.amp.GradScaler(
            "cuda", enabled=not USE_BF16
        )
    except TypeError:
        return torch.cuda.amp.GradScaler(
            enabled=not USE_BF16
        )

def atomic_torch_save(obj, path):
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, temporary_path)
    os.replace(temporary_path, path)

def atomic_numpy_save(array, path):
    path = Path(path)
    temporary_path = path.with_suffix(".tmp.npy")
    np.save(temporary_path, array)
    os.replace(temporary_path, path)

def atomic_npz_save(path, **arrays):
    path = Path(path)
    temporary_path = path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary_path, **arrays)
    os.replace(temporary_path, path)

def atomic_json_save(obj, path):
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")

    with open(temporary_path, "w", encoding="utf-8") as file:
        json.dump(obj, file, ensure_ascii=False, indent=2)

    os.replace(temporary_path, path)

def load_torch(path):
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )
    except TypeError:
        return torch.load(
            path,
            map_location="cpu"
        )

def optimizer_to(optimizer, device):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)

def capture_rng():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all()
    }

def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    torch.cuda.set_rng_state_all(state["cuda"])

tokenizer = AutoTokenizer.from_pretrained(
    ENCODER_NAME,
    use_fast=True
)

tokenizer.save_pretrained(RUN_DIR / "tokenizer")

print("Tokenizer ready.")

Tokenizer ready.


### **Prepare the 66,480-example dialect resource**

In [8]:
dialect_codes = sorted(
    train_gold_df["config"].unique().tolist()
)

dialect_to_id = {
    code: index
    for index, code in enumerate(dialect_codes)
}

if not set(official_dev_df["config"]).issubset(dialect_to_id):
    raise RuntimeError(
        "A DEV dialect has no gold training class."
    )

dialect_texts = (
    train_gold_df["target_arabic"]
    .str.strip()
    .tolist()
)

dialect_labels = (
    train_gold_df["config"]
    .map(dialect_to_id)
    .to_numpy(np.int64)
)

class_counts = np.bincount(
    dialect_labels,
    minlength=len(dialect_codes)
)

class_weights = np.sqrt(
    class_counts.mean() / class_counts
).astype(np.float32)

class DialectDataset(Dataset):
    def __len__(self):
        return len(dialect_texts)

    def __getitem__(self, index):
        return (
            dialect_texts[index],
            int(dialect_labels[index])
        )

class DialectCollator:
    def __call__(self, items):
        texts, labels = zip(*items)

        batch = tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=DIALECT_MAX_LENGTH,
            return_tensors="pt"
        )

        batch["labels"] = torch.tensor(
            labels,
            dtype=torch.long
        )

        return batch

print("Dialect classes:")
print(dict(zip(
    dialect_codes,
    class_counts.tolist()
)))

Dialect classes:
{'EG': 3108, 'JO': 5501, 'LB': 8906, 'MA': 2573, 'MR': 5515, 'OM': 6280, 'PS': 14933, 'SA': 8470, 'SY': 6071, 'TN': 2034, 'YE': 3089}


### **Train or resume the dialect classifier**

In [9]:
DIALECT_DIR = RUN_DIR / "dialect_resource"
DIALECT_DIR.mkdir(parents=True, exist_ok=True)

DIALECT_LAST_PATH = DIALECT_DIR / "checkpoint_last.pt"
DIALECT_FINAL_PATH = DIALECT_DIR / "model_final.pt"

def save_dialect_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    next_batch,
    step
):
    atomic_torch_save({
        "signature": DIALECT_SIGNATURE,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "next_batch": next_batch,
        "step": step,
        "rng": capture_rng(),
        "dialect_codes": dialect_codes
    }, path)

def train_dialect_resource():
    model = DialectClassifier(
        ENCODER_NAME,
        len(dialect_codes)
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=DIALECT_LR,
        weight_decay=WEIGHT_DECAY
    )

    batches_per_epoch = math.ceil(
        len(dialect_texts) / DIALECT_BATCH_SIZE
    )

    updates_per_epoch = math.ceil(
        batches_per_epoch
        / DIALECT_GRAD_ACCUM_STEPS
    )

    total_updates = (
        updates_per_epoch
        * DIALECT_EPOCHS
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        int(total_updates * WARMUP_RATIO),
        total_updates
    )

    scaler = new_scaler()
    start_epoch = 0
    start_batch = 0
    global_step = 0

    if DIALECT_FINAL_PATH.exists():
        state = load_torch(DIALECT_FINAL_PATH)

        if state["signature"] != DIALECT_SIGNATURE:
            raise RuntimeError(
                "Dialect final checkpoint conflicts with "
                "current parameters. Change RUN_NAME."
            )

        model.load_state_dict(state["model"])

        print(
            "Dialect resource already complete:",
            DIALECT_FINAL_PATH
        )

        return model

    if DIALECT_LAST_PATH.exists():
        state = load_torch(DIALECT_LAST_PATH)

        if state["signature"] != DIALECT_SIGNATURE:
            raise RuntimeError(
                "Dialect resume checkpoint conflicts with "
                "current parameters. Change RUN_NAME."
            )

        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        optimizer_to(optimizer, DEVICE)
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])

        start_epoch = state["epoch"]
        start_batch = state["next_batch"]
        global_step = state["step"]

        restore_rng(state["rng"])

        print(
            f"Resuming dialect training at "
            f"epoch={start_epoch}, "
            f"batch={start_batch}, "
            f"step={global_step}"
        )

    weights = torch.tensor(
        class_weights,
        device=DEVICE
    )

    for epoch in range(
        start_epoch,
        DIALECT_EPOCHS
    ):
        generator = torch.Generator()
        generator.manual_seed(SEED + epoch)

        loader = DataLoader(
            DialectDataset(),
            batch_size=DIALECT_BATCH_SIZE,
            shuffle=True,
            generator=generator,
            collate_fn=DialectCollator(),
            num_workers=NUM_WORKERS,
            pin_memory=True
        )

        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        progress = tqdm(
            enumerate(loader),
            total=len(loader),
            desc=(
                f"Dialect epoch "
                f"{epoch + 1}/{DIALECT_EPOCHS}"
            )
        )

        for batch_index, batch in progress:
            if (
                epoch == start_epoch
                and batch_index < start_batch
            ):
                continue

            labels = batch.pop("labels").to(
                DEVICE,
                non_blocking=True
            )

            inputs = {
                key: value.to(
                    DEVICE,
                    non_blocking=True
                )
                for key, value in batch.items()
            }

            with amp_context():
                logits = model(**inputs)

                loss = F.cross_entropy(
                    logits,
                    labels,
                    weight=weights
                )

            scaled_loss = (
                loss
                / DIALECT_GRAD_ACCUM_STEPS
            )

            scaler.scale(
                scaled_loss
            ).backward()

            running_loss += float(
                loss.detach()
            )

            do_step = (
                (batch_index + 1)
                % DIALECT_GRAD_ACCUM_STEPS
                == 0
                or batch_index + 1 == len(loader)
            )

            if do_step:
                scaler.unscale_(optimizer)

                nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM
                )

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1

                progress.set_postfix(
                    loss=(
                        f"{running_loss / max(1, batch_index + 1):.4f}"
                    ),
                    step=global_step
                )

                if (
                    global_step
                    % DIALECT_SAVE_STEPS
                    == 0
                ):
                    save_dialect_checkpoint(
                        DIALECT_LAST_PATH,
                        model,
                        optimizer,
                        scheduler,
                        scaler,
                        epoch,
                        batch_index + 1,
                        global_step
                    )

        start_batch = 0

        save_dialect_checkpoint(
            DIALECT_LAST_PATH,
            model,
            optimizer,
            scheduler,
            scaler,
            epoch + 1,
            0,
            global_step
        )

    atomic_torch_save({
        "signature": DIALECT_SIGNATURE,
        "model": model.state_dict(),
        "dialect_codes": dialect_codes,
        "step": global_step
    }, DIALECT_FINAL_PATH)

    print(
        "Saved dialect resource:",
        DIALECT_FINAL_PATH
    )

    return model

dialect_model = train_dialect_resource()

Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


Dialect resource already complete: /home/mabdallah/alexandriax_mt_14d/rerankers/94_xlmr_large_dialect_crossencoder_pairrank_v1/dialect_resource/model_final.pt


### **Score every candidate for requested-dialect compatibility**

In [10]:
DIALECT_FEATURE_PATH = (
    RUN_DIR
    / "candidate_dialect_features.npz"
)

def compute_dialect_features(model):
    if DIALECT_FEATURE_PATH.exists():
        cache = np.load(
            DIALECT_FEATURE_PATH,
            allow_pickle=False
        )

        if (
            cache["signature"].item()
            == DIALECT_SIGNATURE
            and cache["candidate_hash"].item()
            == CANDIDATE_HASH
        ):
            print(
                "Loaded dialect feature cache:",
                DIALECT_FEATURE_PATH
            )

            return cache["features"]

    model.eval()

    features = np.empty(
        (N * K, 3),
        dtype=np.float32
    )

    flat_texts = (
        candidate_texts
        .reshape(-1)
        .tolist()
    )

    target_ids = np.repeat(
        official_dev_df["config"]
        .map(dialect_to_id)
        .to_numpy(np.int64),
        K
    )

    for start in tqdm(
        range(
            0,
            len(flat_texts),
            SCORE_BATCH_SIZE
        ),
        desc="Candidate dialect scores"
    ):
        end = min(
            start + SCORE_BATCH_SIZE,
            len(flat_texts)
        )

        batch = tokenizer(
            flat_texts[start:end],
            padding=True,
            truncation=True,
            max_length=DIALECT_MAX_LENGTH,
            return_tensors="pt"
        )

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.inference_mode(), amp_context():
            logits = model(**batch).float()

        log_probabilities = logits.log_softmax(-1)
        probabilities = log_probabilities.exp()

        ids = torch.tensor(
            target_ids[start:end],
            device=DEVICE
        )

        row_indices = torch.arange(
            end - start,
            device=DEVICE
        )

        target_log_probability = (
            log_probabilities[
                row_indices,
                ids
            ]
        )

        other_log_probabilities = (
            log_probabilities.clone()
        )

        other_log_probabilities[
            row_indices,
            ids
        ] = -torch.inf

        target_probability = probabilities[
            row_indices,
            ids
        ]

        margin = (
            target_log_probability
            - other_log_probabilities
            .max(-1)
            .values
        ).clamp(-10, 10) / 10

        entropy = -(
            probabilities
            * log_probabilities
        ).sum(-1)

        certainty = (
            1
            - entropy / math.log(
                len(dialect_codes)
            )
        ).clamp(0, 1)

        features[start:end] = torch.stack([
            target_probability.float(),
            margin.float(),
            certainty.float()
        ], dim=-1).cpu().numpy()

    features = features.reshape(
        N,
        K,
        3
    )

    atomic_npz_save(
        DIALECT_FEATURE_PATH,
        features=features,
        signature=np.array(
            DIALECT_SIGNATURE
        ),
        candidate_hash=np.array(
            CANDIDATE_HASH
        )
    )

    print(
        "Saved dialect feature cache:",
        DIALECT_FEATURE_PATH
    )

    return features

dialect_features = compute_dialect_features(
    dialect_model
)

del dialect_model
gc.collect()
torch.cuda.empty_cache()

dialect_diagnostic = pd.DataFrame({
    "variant": variant_names,
    "mean_requested_dialect_probability": (
        dialect_features[:, :, 0]
        .mean(0)
    )
}).sort_values(
    "mean_requested_dialect_probability",
    ascending=False
)

display(dialect_diagnostic)

Loaded dialect feature cache: /home/mabdallah/alexandriax_mt_14d/rerankers/94_xlmr_large_dialect_crossencoder_pairrank_v1/candidate_dialect_features.npz


,variant,mean_requested_dialect_probability
3,03_retrieved_two_shot,0.712610
5,05_retrieved_two_shot_with_participants,0.710283
1,01_exact_training_parity,0.709211
4,04_training_parity_with_participants,0.707199
6,06_ckpt16500_retrieved_two_shot,0.704373
7,07_ckpt16000_retrieved_two_shot,0.696466
2,02_metadata_no_shots,0.695586
0,00_previous_official_control,0.694489
8,08_interp_015_035_050_retrieved_two_shot,0.690959


### **Build exact reference-derived training statistics**

In [11]:
LABEL_CACHE_PATH = (
    RUN_DIR
    / "candidate_metric_statistics.npz"
)

bleu_metric = BLEU(
    tokenize="flores200"
)

chrf_metric = CHRF(
    word_order=2
)

def build_metric_cache():
    if LABEL_CACHE_PATH.exists():
        cache = np.load(
            LABEL_CACHE_PATH,
            allow_pickle=False
        )

        if (
            cache["candidate_hash"].item()
            == CANDIDATE_HASH
        ):
            print(
                "Loaded metric-statistics cache:",
                LABEL_CACHE_PATH
            )

            return (
                cache["bleu_stats"],
                cache["chrf_scores"]
            )

    references = official_dev_df[
        "reference_arabic"
    ].tolist()

    bleu_stats = np.empty(
        (N, K, 10),
        dtype=np.int64
    )

    chrf_scores = np.empty(
        (N, K),
        dtype=np.float32
    )

    for candidate_index, name in enumerate(
        tqdm(
            variant_names,
            desc="Reference-derived label cache"
        )
    ):
        hypotheses = candidate_texts[
            :,
            candidate_index
        ].tolist()

        candidate_bleu_stats = (
            bleu_metric
            ._extract_corpus_statistics(
                hypotheses,
                [references]
            )
        )

        candidate_chrf_stats = (
            chrf_metric
            ._extract_corpus_statistics(
                hypotheses,
                [references]
            )
        )

        bleu_stats[
            :,
            candidate_index
        ] = np.asarray(
            candidate_bleu_stats,
            dtype=np.int64
        )

        chrf_scores[
            :,
            candidate_index
        ] = np.asarray([
            chrf_metric
            ._compute_score_from_stats([
                int(value)
                for value in statistics
            ])
            .score
            for statistics
            in candidate_chrf_stats
        ], dtype=np.float32)

    atomic_npz_save(
        LABEL_CACHE_PATH,
        bleu_stats=bleu_stats,
        chrf_scores=chrf_scores,
        candidate_hash=np.array(
            CANDIDATE_HASH
        )
    )

    print(
        "Saved metric-statistics cache:",
        LABEL_CACHE_PATH
    )

    return bleu_stats, chrf_scores

bleu_stats, chrf_scores = build_metric_cache()

baseline_bleu_stats = bleu_stats[
    np.arange(N),
    baseline_idx
]

baseline_chrf_scores = chrf_scores[
    np.arange(N),
    baseline_idx
]

print("Metric statistics ready.")

Loaded metric-statistics cache: /home/mabdallah/alexandriax_mt_14d/rerankers/94_xlmr_large_dialect_crossencoder_pairrank_v1/candidate_metric_statistics.npz
Metric statistics ready.


### **Build fold-specific winner–loser pairs**

In [12]:
def bleu_from_stats(statistics):
    score = bleu_metric._compute_score_from_stats([
        int(value)
        for value in statistics
    ])

    return float(score.score)

def robust_z(values):
    values = np.asarray(
        values,
        dtype=np.float64
    )

    median = np.median(values)

    scale = 1.4826 * np.median(
        np.abs(values - median)
    )

    if scale < 1e-12:
        scale = values.std()

    if scale < 1e-12:
        return np.zeros_like(values)

    return (
        values - median
    ) / scale

def build_fold_pairs(fold):
    fold_dir = RUN_DIR / f"fold_{fold}"
    fold_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    path = (
        fold_dir
        / "training_pairs.npz"
    )

    if path.exists():
        cache = np.load(
            path,
            allow_pickle=False
        )

        if (
            cache["signature"].item()
            == RANK_SIGNATURE
            and cache["candidate_hash"].item()
            == CANDIDATE_HASH
        ):
            print(
                f"Fold {fold}: loaded "
                f"{len(cache['row_idx']):,} "
                "cached pairs"
            )

            return {
                key: cache[key]
                for key in [
                    "row_idx",
                    "winner_idx",
                    "loser_idx",
                    "weight"
                ]
            }

    training_indices = np.where(
        fold_id != fold
    )[0]

    utility = np.full(
        (N, K),
        np.nan,
        dtype=np.float32
    )

    configurations = official_dev_df[
        "config"
    ].to_numpy()

    for country in sorted(
        set(configurations[training_indices])
    ):
        country_indices = training_indices[
            configurations[training_indices]
            == country
        ]

        baseline_sum = (
            baseline_bleu_stats[
                country_indices
            ]
            .sum(0)
        )

        baseline_score = bleu_from_stats(
            baseline_sum
        )

        marginal_spbleu = np.empty(
            (
                len(country_indices),
                K
            ),
            dtype=np.float64
        )

        for local_index, row_index in enumerate(
            country_indices
        ):
            for candidate_index in range(K):
                replacement_statistics = (
                    baseline_sum
                    - baseline_bleu_stats[
                        row_index
                    ]
                    + bleu_stats[
                        row_index,
                        candidate_index
                    ]
                )

                marginal_spbleu[
                    local_index,
                    candidate_index
                ] = (
                    bleu_from_stats(
                        replacement_statistics
                    )
                    - baseline_score
                )

        chrf_delta = (
            chrf_scores[country_indices]
            - baseline_chrf_scores[
                country_indices,
                None
            ]
        )

        normalized_spbleu = robust_z(
            marginal_spbleu.reshape(-1)
        ).reshape(
            marginal_spbleu.shape
        )

        normalized_chrf = robust_z(
            chrf_delta.reshape(-1)
        ).reshape(
            chrf_delta.shape
        )

        utility[country_indices] = (
            LABEL_SPBLEU_WEIGHT
            * normalized_spbleu
            + LABEL_CHRF_WEIGHT
            * normalized_chrf
        )

    rows = []
    winners = []
    losers = []
    weights = []

    hard_count = (
        PAIRS_PER_TURN // 2
    )

    for row_index in tqdm(
        training_indices,
        desc=f"Fold {fold} pairs"
    ):
        unique_candidates = {}

        for candidate_index in range(K):
            key = normalized_text(
                candidate_texts[
                    row_index,
                    candidate_index
                ]
            )

            unique_candidates.setdefault(
                key,
                candidate_index
            )

        indices = list(
            unique_candidates.values()
        )

        available_pairs = []

        for left in range(len(indices)):
            for right in range(
                left + 1,
                len(indices)
            ):
                first = indices[left]
                second = indices[right]

                gap = abs(float(
                    utility[row_index, first]
                    - utility[row_index, second]
                ))

                if gap < MIN_UTILITY_GAP:
                    continue

                if (
                    utility[row_index, first]
                    > utility[row_index, second]
                ):
                    winner = first
                    loser = second
                else:
                    winner = second
                    loser = first

                available_pairs.append((
                    gap,
                    winner,
                    loser
                ))

        ordered_pairs = sorted(
            available_pairs
        )

        chosen_pairs = ordered_pairs[
            :hard_count
        ]

        used = {
            (winner, loser)
            for _, winner, loser
            in chosen_pairs
        }

        for item in reversed(
            ordered_pairs
        ):
            if (
                len(chosen_pairs)
                >= PAIRS_PER_TURN
            ):
                break

            pair_key = (
                item[1],
                item[2]
            )

            if pair_key not in used:
                chosen_pairs.append(item)
                used.add(pair_key)

        for gap, winner, loser in chosen_pairs:
            rows.append(row_index)
            winners.append(winner)
            losers.append(loser)

            weights.append(np.clip(
                gap,
                PAIR_WEIGHT_MIN,
                PAIR_WEIGHT_MAX
            ))

    pairs = {
        "row_idx": np.asarray(
            rows,
            dtype=np.int32
        ),
        "winner_idx": np.asarray(
            winners,
            dtype=np.int8
        ),
        "loser_idx": np.asarray(
            losers,
            dtype=np.int8
        ),
        "weight": np.asarray(
            weights,
            dtype=np.float32
        )
    }

    atomic_npz_save(
        path,
        **pairs,
        signature=np.array(
            RANK_SIGNATURE
        ),
        candidate_hash=np.array(
            CANDIDATE_HASH
        )
    )

    print(
        f"Fold {fold}: saved "
        f"{len(rows):,} pairs"
    )

    return pairs

### **Construct source-aware quality inputs and batches**

In [13]:
DEV_ROWS = official_dev_df.to_dict(
    "records"
)

def context_string(value):
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except Exception:
            value = []

    if not isinstance(value, list):
        return "None"

    parts = []

    for turn in value:
        if not isinstance(turn, dict):
            continue

        speaker = str(
            turn.get("speaker", "")
        ).strip()

        direction = str(
            turn.get("direction", "")
        ).strip()

        text = str(
            turn.get("text", "")
        ).strip()

        prefix = "/".join(
            item
            for item in [
                speaker,
                direction
            ]
            if item
        )

        if text:
            parts.append(
                f"{prefix}: {text}"
                if prefix
                else text
            )

    return (
        " <turn> ".join(parts)
        if parts
        else "None"
    )

def quality_input(row, candidate):
    code = str(row["config"])
    dialect = DIALECT_NAMES.get(
        code,
        code
    )

    return (
        f"Requested dialect: {code} ({dialect})\n"
        f"Domain: {row.get('domain', '')}\n"
        f"Speaker: {row.get('speaker', '')}\n"
        f"Direction: {row.get('gender_direction', '')}\n"
        f"Previous English turns: "
        f"{context_string(row.get('previous_english_turns', []))}\n"
        f"Current English source: {row['source_text']}\n"
        f"Arabic candidate: {candidate}"
    )

class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(
            self.pairs["row_idx"]
        )

    def __getitem__(self, index):
        return (
            int(
                self.pairs[
                    "row_idx"
                ][index]
            ),
            int(
                self.pairs[
                    "winner_idx"
                ][index]
            ),
            int(
                self.pairs[
                    "loser_idx"
                ][index]
            ),
            float(
                self.pairs[
                    "weight"
                ][index]
            )
        )

class PairCollator:
    def __call__(self, items):
        rows, winners, losers, weights = zip(
            *items
        )

        winner_texts = [
            quality_input(
                DEV_ROWS[row],
                candidate_texts[
                    row,
                    winner
                ]
            )
            for row, winner
            in zip(rows, winners)
        ]

        loser_texts = [
            quality_input(
                DEV_ROWS[row],
                candidate_texts[
                    row,
                    loser
                ]
            )
            for row, loser
            in zip(rows, losers)
        ]

        winner_features = np.stack([
            dialect_features[
                row,
                winner
            ]
            for row, winner
            in zip(rows, winners)
        ])

        loser_features = np.stack([
            dialect_features[
                row,
                loser
            ]
            for row, loser
            in zip(rows, losers)
        ])

        features = np.vstack([
            winner_features,
            loser_features
        ])

        batch = tokenizer(
            winner_texts + loser_texts,
            padding=True,
            truncation=True,
            max_length=RANK_MAX_LENGTH,
            return_tensors="pt"
        )

        batch["dialect_features"] = torch.tensor(
            features,
            dtype=torch.float32
        )

        batch["weights"] = torch.tensor(
            weights,
            dtype=torch.float32
        )

        batch["pair_batch_size"] = len(items)

        return batch

### **Define resumable fold training and held-out scoring**

In [14]:
def encoder_state_from_dialect_checkpoint():
    state = load_torch(
        DIALECT_FINAL_PATH
    )

    if (
        state["signature"]
        != DIALECT_SIGNATURE
    ):
        raise RuntimeError(
            "Dialect checkpoint signature mismatch."
        )

    return {
        key[len("encoder."):]: value
        for key, value
        in state["model"].items()
        if key.startswith("encoder.")
    }

DIALECT_ENCODER_STATE = (
    encoder_state_from_dialect_checkpoint()
)

def save_rank_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    next_batch,
    step
):
    atomic_torch_save({
        "signature": RANK_SIGNATURE,
        "candidate_hash": CANDIDATE_HASH,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "next_batch": next_batch,
        "step": step,
        "rng": capture_rng()
    }, path)

def train_rank_fold(fold, pairs):
    fold_dir = (
        RUN_DIR
        / f"fold_{fold}"
    )

    last_path = (
        fold_dir
        / "checkpoint_last.pt"
    )

    final_path = (
        fold_dir
        / "model_final.pt"
    )

    model = DialectAwareQualityModel(
        ENCODER_NAME
    ).to(DEVICE)

    incompatible = (
        model.encoder
        .load_state_dict(
            DIALECT_ENCODER_STATE,
            strict=True
        )
    )

    if (
        incompatible.missing_keys
        or incompatible.unexpected_keys
    ):
        raise RuntimeError(
            str(incompatible)
        )

    optimizer = torch.optim.AdamW([
        {
            "params": model.encoder.parameters(),
            "lr": RANK_ENCODER_LR
        },
        {
            "params": model.head.parameters(),
            "lr": RANK_HEAD_LR
        }
    ], weight_decay=WEIGHT_DECAY)

    batches_per_epoch = math.ceil(
        len(pairs["row_idx"])
        / RANK_BATCH_SIZE
    )

    updates_per_epoch = math.ceil(
        batches_per_epoch
        / RANK_GRAD_ACCUM_STEPS
    )

    total_updates = (
        updates_per_epoch
        * RANK_EPOCHS
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        int(total_updates * WARMUP_RATIO),
        total_updates
    )

    scaler = new_scaler()
    start_epoch = 0
    start_batch = 0
    global_step = 0

    if final_path.exists():
        state = load_torch(final_path)

        if (
            state["signature"]
            != RANK_SIGNATURE
            or state["candidate_hash"]
            != CANDIDATE_HASH
        ):
            raise RuntimeError(
                f"Fold {fold} final checkpoint "
                "conflicts with the current run. "
                "Change RUN_NAME."
            )

        model.load_state_dict(
            state["model"]
        )

        print(
            f"Fold {fold}: "
            "training already complete"
        )

        return model

    if last_path.exists():
        state = load_torch(last_path)

        if (
            state["signature"]
            != RANK_SIGNATURE
            or state["candidate_hash"]
            != CANDIDATE_HASH
        ):
            raise RuntimeError(
                f"Fold {fold} resume checkpoint "
                "conflicts with the current run. "
                "Change RUN_NAME."
            )

        model.load_state_dict(
            state["model"]
        )

        optimizer.load_state_dict(
            state["optimizer"]
        )

        optimizer_to(
            optimizer,
            DEVICE
        )

        scheduler.load_state_dict(
            state["scheduler"]
        )

        scaler.load_state_dict(
            state["scaler"]
        )

        start_epoch = state["epoch"]
        start_batch = state["next_batch"]
        global_step = state["step"]

        restore_rng(state["rng"])

        print(
            f"Fold {fold}: resume "
            f"epoch={start_epoch}, "
            f"batch={start_batch}, "
            f"step={global_step}"
        )

    dataset = PairDataset(pairs)

    for epoch in range(
        start_epoch,
        RANK_EPOCHS
    ):
        generator = torch.Generator()

        generator.manual_seed(
            SEED
            + 1000 * fold
            + epoch
        )

        loader = DataLoader(
            dataset,
            batch_size=RANK_BATCH_SIZE,
            shuffle=True,
            generator=generator,
            collate_fn=PairCollator(),
            num_workers=NUM_WORKERS,
            pin_memory=True
        )

        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        progress = tqdm(
            enumerate(loader),
            total=len(loader),
            desc=(
                f"Fold {fold} rank epoch "
                f"{epoch + 1}/{RANK_EPOCHS}"
            )
        )

        for batch_index, batch in progress:
            if (
                epoch == start_epoch
                and batch_index < start_batch
            ):
                continue

            pair_batch_size = batch.pop(
                "pair_batch_size"
            )

            weights = batch.pop(
                "weights"
            ).to(DEVICE)

            inputs = {
                key: value.to(
                    DEVICE,
                    non_blocking=True
                )
                for key, value
                in batch.items()
            }

            with amp_context():
                scores = model(**inputs)

                winner_scores = scores[
                    :pair_batch_size
                ]

                loser_scores = scores[
                    pair_batch_size:
                ]

                pair_losses = F.softplus(
                    -(
                        winner_scores
                        - loser_scores
                    )
                    / RANK_TEMPERATURE
                )

                loss = (
                    pair_losses
                    * weights
                ).sum() / weights.sum()

            scaled_loss = (
                loss
                / RANK_GRAD_ACCUM_STEPS
            )

            scaler.scale(
                scaled_loss
            ).backward()

            running_loss += float(
                loss.detach()
            )

            do_step = (
                (batch_index + 1)
                % RANK_GRAD_ACCUM_STEPS
                == 0
                or batch_index + 1
                == len(loader)
            )

            if do_step:
                scaler.unscale_(optimizer)

                nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM
                )

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1

                progress.set_postfix(
                    loss=(
                        f"{running_loss / max(1, batch_index + 1):.4f}"
                    ),
                    step=global_step
                )

                if (
                    global_step
                    % RANK_SAVE_STEPS
                    == 0
                ):
                    save_rank_checkpoint(
                        last_path,
                        model,
                        optimizer,
                        scheduler,
                        scaler,
                        epoch,
                        batch_index + 1,
                        global_step
                    )

        start_batch = 0

        save_rank_checkpoint(
            last_path,
            model,
            optimizer,
            scheduler,
            scaler,
            epoch + 1,
            0,
            global_step
        )

    atomic_torch_save({
        "signature": RANK_SIGNATURE,
        "candidate_hash": CANDIDATE_HASH,
        "model": model.state_dict(),
        "step": global_step
    }, final_path)

    print(
        f"Fold {fold}: saved final model to "
        f"{final_path}"
    )

    return model

def score_fold(fold, model):
    fold_dir = (
        RUN_DIR
        / f"fold_{fold}"
    )

    score_path = (
        fold_dir
        / "heldout_candidate_scores.npy"
    )

    metadata_path = (
        fold_dir
        / "heldout_candidate_scores.json"
    )

    validation_indices = np.where(
        fold_id == fold
    )[0]

    expected_ids = official_dev_df.loc[
        validation_indices,
        "source_id"
    ].tolist()

    if (
        score_path.exists()
        and metadata_path.exists()
    ):
        metadata = json.loads(
            metadata_path.read_text(
                encoding="utf-8"
            )
        )

        if (
            metadata["signature"]
            != RANK_SIGNATURE
            or metadata["candidate_hash"]
            != CANDIDATE_HASH
            or metadata["source_ids"]
            != expected_ids
        ):
            raise RuntimeError(
                f"Fold {fold} score cache mismatch."
            )

        scores = np.load(score_path)

        if scores.shape != (
            len(validation_indices),
            K
        ):
            raise RuntimeError(
                f"Fold {fold} score-cache "
                "shape mismatch."
            )

    else:
        scores = np.full(
            (
                len(validation_indices),
                K
            ),
            np.nan,
            dtype=np.float32
        )

        atomic_json_save({
            "signature": RANK_SIGNATURE,
            "candidate_hash": CANDIDATE_HASH,
            "source_ids": expected_ids
        }, metadata_path)

    pending = np.where(
        ~np.isfinite(scores).all(1)
    )[0]

    model.eval()

    for start in tqdm(
        range(
            0,
            len(pending),
            SCORE_SAVE_ROWS
        ),
        desc=f"Fold {fold} held-out scoring"
    ):
        local_positions = pending[
            start:start + SCORE_SAVE_ROWS
        ]

        row_indices = validation_indices[
            local_positions
        ]

        texts = [
            quality_input(
                DEV_ROWS[row_index],
                candidate_texts[
                    row_index,
                    candidate_index
                ]
            )
            for row_index in row_indices
            for candidate_index in range(K)
        ]

        features = np.vstack([
            dialect_features[
                row_index,
                candidate_index
            ]
            for row_index in row_indices
            for candidate_index in range(K)
        ])

        chunk_scores = []

        for batch_start in range(
            0,
            len(texts),
            SCORE_BATCH_SIZE
        ):
            batch_end = min(
                batch_start + SCORE_BATCH_SIZE,
                len(texts)
            )

            batch = tokenizer(
                texts[
                    batch_start:batch_end
                ],
                padding=True,
                truncation=True,
                max_length=RANK_MAX_LENGTH,
                return_tensors="pt"
            )

            batch = {
                key: value.to(DEVICE)
                for key, value
                in batch.items()
            }

            feature_batch = torch.tensor(
                features[
                    batch_start:batch_end
                ],
                dtype=torch.float32,
                device=DEVICE
            )

            with torch.inference_mode(), amp_context():
                batch_scores = model(
                    **batch,
                    dialect_features=feature_batch
                ).float().cpu().numpy()

            chunk_scores.append(
                batch_scores
            )

        scores[local_positions] = (
            np.concatenate(
                chunk_scores
            )
            .reshape(
                len(local_positions),
                K
            )
        )

        atomic_numpy_save(
            scores,
            score_path
        )

    if not np.isfinite(scores).all():
        raise RuntimeError(
            f"Fold {fold} scoring is incomplete."
        )

    return validation_indices, scores

### **Run or resume all five OOF folds**

In [ ]:
oof_scores = np.full(
    (N, K),
    np.nan,
    dtype=np.float32
)

for fold in range(N_FOLDS):
    print("\n" + "=" * 90)
    print(
        f"OUTER FOLD "
        f"{fold + 1}/{N_FOLDS}"
    )
    print("=" * 90)

    fold_pairs = build_fold_pairs(
        fold
    )

    rank_model = train_rank_fold(
        fold,
        fold_pairs
    )

    validation_indices, validation_scores = (
        score_fold(
            fold,
            rank_model
        )
    )

    oof_scores[
        validation_indices
    ] = validation_scores

    del rank_model
    del fold_pairs

    gc.collect()
    torch.cuda.empty_cache()

if not np.isfinite(oof_scores).all():
    raise RuntimeError(
        "OOF score matrix is incomplete."
    )

OOF_SCORE_PATH = (
    OUTPUT_DIR
    / "oof_candidate_scores.npy"
)

atomic_numpy_save(
    oof_scores,
    OOF_SCORE_PATH
)

print(
    "Saved complete OOF score matrix:",
    OOF_SCORE_PATH
)


OUTER FOLD 1/5
Fold 0: loaded 48,174 cached pairs
Fold 0: training already complete


Fold 0 held-out scoring: 0it [00:00, ?it/s]


OUTER FOLD 2/5
Fold 1: loaded 48,261 cached pairs
Fold 1: training already complete


Fold 1 held-out scoring: 0it [00:00, ?it/s]


OUTER FOLD 3/5
Fold 2: loaded 48,084 cached pairs


### **Select the highest-scoring candidate**

In [ ]:
best_idx = oof_scores.argmax(1)

sorted_scores = np.sort(
    oof_scores,
    axis=1
)

score_margin = (
    sorted_scores[:, -1]
    - sorted_scores[:, -2]
)

fallback = (
    score_margin
    <= MIN_SCORE_MARGIN
)

selected_idx = np.where(
    fallback,
    baseline_idx,
    best_idx
)

selected_predictions = candidate_texts[
    np.arange(N),
    selected_idx
]

selected_variants = np.asarray(
    variant_names,
    dtype=object
)[selected_idx]

baseline_variants = np.asarray(
    variant_names,
    dtype=object
)[baseline_idx]

reranker_turn_df = official_dev_df[[
    "source_id",
    "config",
    "country",
    "conversation_id",
    "turn_order",
    "source_text"
]].copy()

reranker_turn_df["prediction"] = (
    selected_predictions
)

reranker_turn_df[
    "selected_from_variant"
] = selected_variants

reranker_turn_df[
    "selected_candidate_index"
] = selected_idx

reranker_turn_df[
    "selected_score"
] = oof_scores[
    np.arange(N),
    selected_idx
]

reranker_turn_df[
    "runner_up_margin"
] = score_margin

reranker_turn_df[
    "requested_dialect_probability"
] = dialect_features[
    np.arange(N),
    selected_idx,
    0
]

reranker_turn_df[
    "baseline_from_variant"
] = baseline_variants

reranker_turn_df[
    "baseline_candidate_index"
] = baseline_idx

reranker_turn_df[
    "changed_from_system92"
] = [
    normalized_text(selected)
    != normalized_text(baseline)
    for selected, baseline
    in zip(
        selected_predictions,
        baseline_predictions
    )
]

RERANKER_TURN_PATH = (
    OUTPUT_DIR
    / "reranker_oof_turn_predictions.csv"
)

reranker_turn_df.to_csv(
    RERANKER_TURN_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Text-level overrides:",
    int(
        reranker_turn_df[
            "changed_from_system92"
        ].sum()
    ),
    "/",
    N
)

display(pd.crosstab(
    reranker_turn_df["config"],
    reranker_turn_df[
        "selected_from_variant"
    ],
    margins=True
))

### **Calculate official DEV metrics and choose the deployable system**

In [ ]:
def official_metrics(
    predictions,
    system_name
):
    scored = official_dev_df[[
        "source_id",
        "config",
        "country",
        "conversation_id",
        "turn_order",
        "source_text",
        "reference_arabic"
    ]].copy()

    scored["prediction"] = np.asarray(
        predictions,
        dtype=object
    )

    rows = []

    for country in sorted(
        scored["config"].unique()
    ):
        country_df = scored[
            scored["config"] == country
        ]

        hypotheses = country_df[
            "prediction"
        ].astype(str).tolist()

        references = country_df[
            "reference_arabic"
        ].astype(str).tolist()

        rows.append({
            "country": country,
            "turns": len(country_df),
            "spBLEU": sacrebleu.corpus_bleu(
                hypotheses,
                [references],
                tokenize="flores200"
            ).score,
            "chrF++": sacrebleu.corpus_chrf(
                hypotheses,
                [references],
                word_order=2
            ).score,
            "BLEU": sacrebleu.corpus_bleu(
                hypotheses,
                [references]
            ).score,
            "chrF": sacrebleu.corpus_chrf(
                hypotheses,
                [references],
                word_order=0
            ).score
        })

    per_country = pd.DataFrame(rows)

    summary = {
        "system": system_name,
        "Average spBLEU (primary)": float(
            per_country["spBLEU"].mean()
        ),
        "Average chrF++": float(
            per_country["chrF++"].mean()
        )
    }

    return (
        summary,
        per_country,
        scored
    )

reranker_summary, reranker_per_country, reranker_scored = official_metrics(
    selected_predictions,
    RUN_NAME + "_OOF"
)

baseline_summary, baseline_per_country, baseline_scored = official_metrics(
    baseline_predictions,
    BASELINE_VARIANT
)

comparison = pd.DataFrame([
    baseline_summary,
    reranker_summary
])

display(comparison)

country_comparison = baseline_per_country.merge(
    reranker_per_country,
    on=["country", "turns"],
    suffixes=(
        "_system92",
        "_reranker"
    )
)

country_comparison["spBLEU_delta"] = (
    country_comparison["spBLEU_reranker"]
    - country_comparison["spBLEU_system92"]
)

country_comparison["chrF++_delta"] = (
    country_comparison["chrF++_reranker"]
    - country_comparison["chrF++_system92"]
)

display(country_comparison)

reranker_gain = (
    reranker_summary[
        "Average spBLEU (primary)"
    ]
    - baseline_summary[
        "Average spBLEU (primary)"
    ]
)

deploy_reranker = (
    reranker_gain
    > DEPLOY_MIN_SPBLEU_GAIN
)

final_system = (
    RUN_NAME + "_OOF"
    if deploy_reranker
    else BASELINE_VARIANT
)

final_predictions = (
    selected_predictions
    if deploy_reranker
    else baseline_predictions
)

final_summary, final_per_country, final_scored = official_metrics(
    final_predictions,
    final_system
)

comparison.to_csv(
    OUTPUT_DIR
    / "dev_system_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

country_comparison.to_csv(
    OUTPUT_DIR
    / "dev_per_country_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

atomic_json_save({
    "reranker_gain_spBLEU": reranker_gain,
    "minimum_required_gain": DEPLOY_MIN_SPBLEU_GAIN,
    "deployed_system": final_system,
    "reranker_summary": reranker_summary,
    "baseline_summary": baseline_summary
}, OUTPUT_DIR / "deployment_decision.json")

print(
    f"Deployment decision: {final_system}"
)

print(
    f"OOF spBLEU delta: "
    f"{reranker_gain:+.6f}"
)

### **Write official files and submission ZIP**

In [ ]:
submission_turn_df = (
    reranker_turn_df.copy()
)

if not deploy_reranker:
    submission_turn_df[
        "prediction"
    ] = baseline_predictions

    submission_turn_df[
        "selected_from_variant"
    ] = baseline_variants

    submission_turn_df[
        "selected_candidate_index"
    ] = baseline_idx

submission_turn_df.to_csv(
    OUTPUT_DIR
    / "turn_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

final_scored.to_csv(
    OUTPUT_DIR
    / "scored_turn_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

final_per_country.to_csv(
    OUTPUT_DIR
    / "per_country_official_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

score_row = {
    "Variant": final_system,
    "Checkpoint": "5-fold OOF",
    **{
        key: value
        for key, value
        in final_summary.items()
        if key != "system"
    }
}

for row in final_per_country.to_dict(
    "records"
):
    score_row[
        f"{row['country']} spBLEU"
    ] = row["spBLEU"]

    score_row[
        f"{row['country']} chrF++"
    ] = row["chrF++"]

pd.DataFrame([
    score_row
]).to_csv(
    OUTPUT_DIR
    / "official_leaderboard_score_row.csv",
    index=False,
    encoding="utf-8-sig"
)

metrics = {
    "variant_name": final_system,
    "evaluation": (
        "conversation-grouped "
        "5-fold OOF"
    ),
    "num_turns": N,
    "num_countries": EXPECTED_DEV_COUNTRIES,
    **{
        key: value
        for key, value
        in final_summary.items()
        if key != "system"
    },
    "per_country": (
        final_per_country
        .to_dict("records")
    ),
    "reranker_spBLEU_gain_over_system92": (
        reranker_gain
    ),
    "candidate_variants": variant_names,
    "encoder": ENCODER_NAME,
    "sacrebleu_version": sacrebleu.__version__,
    "spbleu_tokenizer": "flores200",
    "chrf_word_order": 2
}

atomic_json_save(
    metrics,
    OUTPUT_DIR
    / "official_metrics.json"
)

submission_records = []

ordered = final_scored.sort_values([
    "config",
    "conversation_id",
    "turn_order"
])

for (
    country,
    conversation_id
), conversation_df in ordered.groupby(
    ["config", "conversation_id"],
    sort=True
):
    if conversation_df[
        "turn_order"
    ].duplicated().any():
        raise RuntimeError(
            f"Duplicate turn order in "
            f"{country}/{conversation_id}"
        )

    turns = [
        {
            "turn_order": int(
                row.turn_order
            ),
            "prediction": str(
                row.prediction
            )
        }
        for row
        in conversation_df.itertuples()
    ]

    submission_records.append({
        "conv_id": str(
            conversation_id
        ),
        "country": str(country),
        "turns": turns
    })

jsonl_path = (
    OUTPUT_DIR
    / "predictions.jsonl"
)

with open(
    jsonl_path,
    "w",
    encoding="utf-8"
) as file:
    for record in submission_records:
        file.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )

zip_path = (
    OUTPUT_DIR
    / "submission_predictions.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:
    zip_file.write(
        jsonl_path,
        arcname="predictions.jsonl"
    )

readback_keys = set()
readback_turns = 0

with zipfile.ZipFile(
    zip_path
) as zip_file:
    if zip_file.namelist() != [
        "predictions.jsonl"
    ]:
        raise RuntimeError(
            "ZIP must contain only "
            "predictions.jsonl"
        )

    with zip_file.open(
        "predictions.jsonl"
    ) as file:
        for line in file:
            record = json.loads(
                line.decode("utf-8")
            )

            for turn in record["turns"]:
                readback_turns += 1

                readback_keys.add((
                    str(record["country"]),
                    str(record["conv_id"]),
                    int(turn["turn_order"])
                ))

expected_keys = set(zip(
    official_dev_df["config"],
    official_dev_df[
        "conversation_id"
    ],
    official_dev_df[
        "turn_order"
    ]
))

if (
    readback_turns
    != EXPECTED_DEV_TURNS
    or readback_keys
    != expected_keys
):
    raise RuntimeError(
        "Submission ZIP readback "
        "validation failed."
    )

print("\nOFFICIAL-STYLE DEVELOPMENT RESULT")
print("System:", final_system)

print(
    "Average spBLEU:",
    f"{final_summary['Average spBLEU (primary)']:.6f}"
)

print(
    "Average chrF++:",
    f"{final_summary['Average chrF++']:.6f}"
)

print("\nSUBMIT THIS ZIP:")
print(zip_path)

display(final_per_country)

# **Qwen3-Reranker-0.6B**

In [22]:
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.51,<5", "peft>=0.17,<1", "accelerate>=1,<2", "huggingface_hub>=0.30", "sentencepiece"
])

print("Dependencies are ready. Restart the kernel, then run the original notebook cells 1-6.")

Dependencies are ready. Restart the kernel, then run the original notebook cells 1-6.


In [34]:
# ============================================================
# EDIT PATHS AND TRAINING PARAMETERS ONLY IN THIS CELL
# This section must be run after the original notebook cells 1-6.
# ============================================================
import contextlib
from huggingface_hub import snapshot_download
from peft import LoraConfig, TaskType, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict
from sklearn.model_selection import GroupShuffleSplit
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

RUN_NAME = "95_qwen3_06b_selective_uplift_cleanlabels_v1"
RUN_DIR = PROJECT_DIR / "rerankers" / RUN_NAME
OUTPUT_DIR = INFERENCE_VARIANTS_ROOT / RUN_NAME
MODEL_ROOT = PROJECT_DIR / "models" / "hf"

TEACHER_REPO = "Unbabel/M-Prometheus-3B"
RERANKER_REPO = "Qwen/Qwen3-Reranker-0.6B"
TEACHER_DIR = MODEL_ROOT / "M-Prometheus-3B"
RERANKER_DIR = MODEL_ROOT / "Qwen3-Reranker-0.6B"

# ---------------- ORACLE AND CLEAN-LABEL PARAMETERS ----------------
MIN_ORACLE_GAIN_TO_CONTINUE = 0.25
ORACLE_MAX_SWEEPS = 5
BLEU_STRONG_GAP = 3.0
CHRF_STRONG_GAP = 2.0
BLEU_TIE_GAP = 1.0
CHRF_TIE_GAP = 1.0
MAX_PAIRS_PER_TURN = 3

# ---------------- TEACHER PARAMETERS ----------------
TEACHER_MAX_LENGTH = 1024
TEACHER_MAX_NEW_TOKENS = 24
TEACHER_PAIR_BATCH_SIZE = 4

# ---------------- QWEN3 TRAINING PARAMETERS ----------------
RANK_EPOCHS = 3
RANK_MAX_LENGTH = 512
RANK_BATCH_SIZE = 4
RANK_GRAD_ACCUM_STEPS = 8
RANK_LR = 1e-4
RANK_WARMUP_RATIO = 0.06
RANK_WEIGHT_DECAY = 0.01
RANK_MAX_GRAD_NORM = 1.0
RANK_SAVE_STEPS = 250
RANK_TEMPERATURE = 1.0
TIE_SCORE_MARGIN = 0.10
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# ---------------- CALIBRATION AND SCORING PARAMETERS ----------------
CALIBRATION_FRACTION = 0.12
CALIBRATION_THRESHOLD_POINTS = 51
CALIBRATION_MIN_OVERRIDES = 10
SCORE_BATCH_SIZE = 16
SCORE_SAVE_ROWS = 128
NUM_WORKERS = 0
DEPLOY_MIN_SPBLEU_GAIN = 0.0
NO_OVERRIDE_THRESHOLD = 1e9

RUN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required.")

DEVICE = torch.device("cuda")
USE_BF16 = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:20]

RUN_SIGNATURE = stable_hash({
    "run": RUN_NAME, "candidate_hash": CANDIDATE_HASH, "teacher": TEACHER_REPO, "reranker": RERANKER_REPO,
    "metric_gaps": [BLEU_STRONG_GAP, CHRF_STRONG_GAP, BLEU_TIE_GAP, CHRF_TIE_GAP],
    "epochs": RANK_EPOCHS, "length": RANK_MAX_LENGTH, "batch": RANK_BATCH_SIZE,
    "accum": RANK_GRAD_ACCUM_STEPS, "lr": RANK_LR, "lora": [LORA_R, LORA_ALPHA, LORA_DROPOUT],
    "teacher_length": TEACHER_MAX_LENGTH, "calibration": [CALIBRATION_FRACTION, CALIBRATION_THRESHOLD_POINTS],
    "folds": N_FOLDS, "seed": SEED
})

def amp_context():
    return torch.autocast("cuda", dtype=AMP_DTYPE)

def new_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=not USE_BF16)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=not USE_BF16)

def atomic_torch_save(obj, path):
    path = Path(path); temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, temporary); os.replace(temporary, path)

def atomic_numpy_save(array, path):
    path = Path(path); temporary = path.with_suffix(".tmp.npy")
    np.save(temporary, array); os.replace(temporary, path)

def atomic_npz_save(path, **arrays):
    path = Path(path); temporary = path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary, **arrays); os.replace(temporary, path)

def atomic_json_save(obj, path):
    path = Path(path); temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8"); os.replace(temporary, path)

def load_torch(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def optimizer_to(optimizer, device):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value): state[key] = value.to(device)

def capture_rng():
    return {
        "python": random.getstate(), "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all()
    }

def restore_rng(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"]); torch.cuda.set_rng_state_all(state["cuda"])

print("Run:", RUN_NAME)
print("Signature:", RUN_SIGNATURE)
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", AMP_DTYPE)
print("TRAINING PARAMETERS: RANK_EPOCHS=3, batch=4, grad_accum=8, LR=1e-4, LoRA r=16")

Run: 95_qwen3_06b_selective_uplift_cleanlabels_v1
Signature: d92fea8fe01c2e22aca5
GPU: NVIDIA GeForce RTX 5090 | dtype: torch.bfloat16
TRAINING PARAMETERS: RANK_EPOCHS=3, batch=4, grad_accum=8, LR=1e-4, LoRA r=16


In [24]:
def ensure_model(repo_id, local_dir):
    local_dir = Path(local_dir)

    if not (local_dir / "config.json").exists():
        print("Downloading", repo_id, "to", local_dir)
        snapshot_download(
            repo_id=repo_id,
            local_dir=str(local_dir),
            ignore_patterns=["*.h5", "*.msgpack", "*.ot"]
        )
    else:
        print("Using local model:", local_dir)

    return str(local_dir)

TEACHER_PATH = ensure_model(TEACHER_REPO, TEACHER_DIR)
RERANKER_PATH = ensure_model(RERANKER_REPO, RERANKER_DIR)

Using local model: /home/mabdallah/alexandriax_mt_14d/models/hf/M-Prometheus-3B
Using local model: /home/mabdallah/alexandriax_mt_14d/models/hf/Qwen3-Reranker-0.6B


In [25]:
METRIC_CACHE_PATH = RUN_DIR / "candidate_metric_statistics.npz"
bleu_metric = BLEU(tokenize="flores200")
sentence_bleu_metric = BLEU(tokenize="flores200", effective_order=True)
chrf_metric = CHRF(word_order=2)

def score_bleu_stats(statistics):
    return float(bleu_metric._compute_score_from_stats([int(value) for value in statistics]).score)

def score_sentence_bleu_stats(statistics):
    return float(sentence_bleu_metric._compute_score_from_stats([int(value) for value in statistics]).score)

def build_metric_cache():
    if METRIC_CACHE_PATH.exists():
        cache = np.load(METRIC_CACHE_PATH, allow_pickle=False)

        if cache["candidate_hash"].item() == CANDIDATE_HASH:
            print("Loaded metric cache:", METRIC_CACHE_PATH)
            return cache["bleu_stats"], cache["sentence_bleu"], cache["chrf_scores"]

    references = official_dev_df["reference_arabic"].astype(str).tolist()
    bleu_stats = np.empty((N, K, 10), dtype=np.int64)
    sentence_bleu = np.empty((N, K), dtype=np.float32)
    chrf_scores = np.empty((N, K), dtype=np.float32)

    for candidate_index, name in enumerate(tqdm(variant_names, desc="Reference metric cache")):
        hypotheses = candidate_texts[:, candidate_index].astype(str).tolist()
        bstats = np.asarray(
            bleu_metric._extract_corpus_statistics(hypotheses, [references]),
            dtype=np.int64
        )
        cstats = chrf_metric._extract_corpus_statistics(hypotheses, [references])

        bleu_stats[:, candidate_index] = bstats
        sentence_bleu[:, candidate_index] = np.asarray(
            [score_sentence_bleu_stats(stat) for stat in bstats],
            dtype=np.float32
        )
        chrf_scores[:, candidate_index] = np.asarray([
            chrf_metric._compute_score_from_stats([int(value) for value in stat]).score
            for stat in cstats
        ], dtype=np.float32)

    atomic_npz_save(
        METRIC_CACHE_PATH,
        bleu_stats=bleu_stats,
        sentence_bleu=sentence_bleu,
        chrf_scores=chrf_scores,
        candidate_hash=np.array(CANDIDATE_HASH)
    )

    return bleu_stats, sentence_bleu, chrf_scores

def official_metrics_for_indices(predictions, indices, system_name):
    indices = np.asarray(indices, dtype=np.int64)
    frame = official_dev_df.iloc[indices][[
        "source_id", "config", "country", "conversation_id",
        "turn_order", "source_text", "reference_arabic"
    ]].copy()
    frame["prediction"] = np.asarray(predictions, dtype=object)
    rows = []

    for country in sorted(frame["config"].unique()):
        country_df = frame[frame["config"] == country]
        hypotheses = country_df["prediction"].astype(str).tolist()
        references = country_df["reference_arabic"].astype(str).tolist()

        rows.append({
            "country": country,
            "turns": len(country_df),
            "spBLEU": sacrebleu.corpus_bleu(
                hypotheses, [references], tokenize="flores200"
            ).score,
            "chrF++": sacrebleu.corpus_chrf(
                hypotheses, [references], word_order=2
            ).score,
            "BLEU": sacrebleu.corpus_bleu(hypotheses, [references]).score,
            "chrF": sacrebleu.corpus_chrf(
                hypotheses, [references], word_order=0
            ).score
        })

    per_country = pd.DataFrame(rows)
    summary = {
        "system": system_name,
        "Average spBLEU (primary)": float(per_country["spBLEU"].mean()),
        "Average chrF++": float(per_country["chrF++"].mean())
    }

    return summary, per_country, frame

def greedy_reference_oracle(max_sweeps=5):
    selected = baseline_idx.copy()
    configurations = official_dev_df["config"].to_numpy()

    for country in sorted(set(configurations)):
        rows = np.where(configurations == country)[0]
        total_stats = bleu_stats[rows, selected[rows]].sum(0)
        current_score = score_bleu_stats(total_stats)

        for sweep in range(max_sweeps):
            changes = 0

            for row_index in rows:
                current_index = int(selected[row_index])
                best_index, best_score = current_index, current_score

                for candidate_index in range(K):
                    if candidate_index == current_index:
                        continue

                    trial = (
                        total_stats
                        - bleu_stats[row_index, current_index]
                        + bleu_stats[row_index, candidate_index]
                    )
                    trial_score = score_bleu_stats(trial)

                    if trial_score > best_score + 1e-12:
                        best_index, best_score = candidate_index, trial_score

                if best_index != current_index:
                    total_stats += (
                        bleu_stats[row_index, best_index]
                        - bleu_stats[row_index, current_index]
                    )
                    selected[row_index] = best_index
                    current_score = best_score
                    changes += 1

            print(
                f"Oracle {country} sweep {sweep + 1}: "
                f"{current_score:.4f}, changes={changes}"
            )

            if changes == 0:
                break

    return selected

bleu_stats, sentence_bleu, chrf_scores = build_metric_cache()
baseline_bleu_stats = bleu_stats[np.arange(N), baseline_idx]
baseline_chrf_scores = chrf_scores[np.arange(N), baseline_idx]

baseline_summary, baseline_per_country, _ = official_metrics_for_indices(
    baseline_predictions, np.arange(N), BASELINE_VARIANT
)

oracle_idx = greedy_reference_oracle(ORACLE_MAX_SWEEPS)
oracle_predictions = candidate_texts[np.arange(N), oracle_idx]

oracle_summary, oracle_per_country, _ = official_metrics_for_indices(
    oracle_predictions, np.arange(N), "greedy_reference_oracle"
)

oracle_gain = (
    oracle_summary["Average spBLEU (primary)"]
    - baseline_summary["Average spBLEU (primary)"]
)

display(pd.DataFrame([baseline_summary, oracle_summary]))

oracle_per_country.to_csv(
    OUTPUT_DIR / "oracle_per_country.csv",
    index=False,
    encoding="utf-8-sig"
)

atomic_json_save({
    "baseline": baseline_summary,
    "oracle": oracle_summary,
    "oracle_gain": oracle_gain
}, OUTPUT_DIR / "oracle_summary.json")

if abs(baseline_summary["Average spBLEU (primary)"] - 30.928003) > 0.02:
    raise RuntimeError("System92 reproduction mismatch; stop before training.")

if oracle_gain < MIN_ORACLE_GAIN_TO_CONTINUE:
    raise RuntimeError(
        f"Candidate-pool oracle gain is only {oracle_gain:+.4f}; "
        "improve generation before reranking."
    )

print(f"Oracle gate passed: {oracle_gain:+.4f} spBLEU available in this pool.")

Loaded metric cache: /home/mabdallah/alexandriax_mt_14d/rerankers/95_qwen3_06b_selective_uplift_cleanlabels_v1/candidate_metric_statistics.npz
Oracle EG sweep 1: 39.0740, changes=767
Oracle EG sweep 2: 39.0776, changes=14
Oracle EG sweep 3: 39.0776, changes=0
Oracle JO sweep 1: 42.0259, changes=752
Oracle JO sweep 2: 42.0268, changes=5
Oracle JO sweep 3: 42.0268, changes=0
Oracle LB sweep 1: 37.6188, changes=735
Oracle LB sweep 2: 37.6376, changes=20
Oracle LB sweep 3: 37.6376, changes=0
Oracle MA sweep 1: 29.0334, changes=847
Oracle MA sweep 2: 29.0510, changes=19
Oracle MA sweep 3: 29.0510, changes=0
Oracle MR sweep 1: 22.5786, changes=801
Oracle MR sweep 2: 22.5903, changes=22
Oracle MR sweep 3: 22.5903, changes=0
Oracle OM sweep 1: 42.1791, changes=784
Oracle OM sweep 2: 42.1793, changes=3
Oracle OM sweep 3: 42.1793, changes=0
Oracle PS sweep 1: 38.8710, changes=752
Oracle PS sweep 2: 38.8711, changes=2
Oracle PS sweep 3: 38.8711, changes=0
Oracle SA sweep 1: 38.9945, changes=788
O

,system,Average spBLEU (primary),Average chrF++
0,92_mixed_best_checkpoint_variant_per_country,30.928003,45.594526
1,greedy_reference_oracle,36.779164,49.933007


Oracle gate passed: +5.8512 spBLEU available in this pool.


In [26]:
PAIR_CANDIDATE_PATH = RUN_DIR / "metric_prefilter_pairs.csv"

def metric_label(bleu_gap, chrf_gap):
    if bleu_gap > 0 and chrf_gap > 0 and (
        bleu_gap >= BLEU_STRONG_GAP or chrf_gap >= CHRF_STRONG_GAP
    ):
        return 1

    if bleu_gap < 0 and chrf_gap < 0 and (
        -bleu_gap >= BLEU_STRONG_GAP or -chrf_gap >= CHRF_STRONG_GAP
    ):
        return -1

    if (
        abs(bleu_gap) <= BLEU_TIE_GAP
        and abs(chrf_gap) <= CHRF_TIE_GAP
    ):
        return 0

    return None

def build_metric_prefilter():
    rows = []

    for row_index in tqdm(range(N), desc="Metric prefilter"):
        base_index = int(baseline_idx[row_index])
        base_text = normalized_text(candidate_texts[row_index, base_index])
        choices = {-1: [], 0: [], 1: []}
        seen = {base_text}

        for alt_index in range(K):
            alt_text = normalized_text(candidate_texts[row_index, alt_index])

            if alt_text in seen:
                continue

            seen.add(alt_text)

            bleu_gap = float(
                sentence_bleu[row_index, alt_index]
                - sentence_bleu[row_index, base_index]
            )
            chrf_gap = float(
                chrf_scores[row_index, alt_index]
                - chrf_scores[row_index, base_index]
            )

            label = metric_label(bleu_gap, chrf_gap)

            if label is None:
                continue

            strength = (
                abs(bleu_gap) / BLEU_STRONG_GAP
                + abs(chrf_gap) / CHRF_STRONG_GAP
            )

            choices[label].append((
                strength, alt_index, bleu_gap, chrf_gap
            ))

        for label in [-1, 0, 1]:
            if not choices[label]:
                continue

            ordered = sorted(
                choices[label],
                reverse=label != 0
            )
            strength, alt_index, bleu_gap, chrf_gap = ordered[0]
            pair_id = stable_hash([
                CANDIDATE_HASH, row_index,
                int(base_index), int(alt_index)
            ])

            rows.append({
                "pair_id": pair_id,
                "row_idx": row_index,
                "config": official_dev_df.loc[row_index, "config"],
                "baseline_idx": base_index,
                "alt_idx": alt_index,
                "metric_label": label,
                "bleu_gap": bleu_gap,
                "chrf_gap": chrf_gap,
                "strength": strength,
                "requires_teacher": label != 0
            })

    result = pd.DataFrame(rows).sort_values([
        "row_idx", "metric_label"
    ]).reset_index(drop=True)

    if result.groupby("row_idx").size().max() > MAX_PAIRS_PER_TURN:
        raise RuntimeError("Pair cap violated.")

    result.to_csv(
        PAIR_CANDIDATE_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    return result

pair_candidates = build_metric_prefilter()

display(pd.crosstab(
    pair_candidates["config"],
    pair_candidates["metric_label"],
    margins=True
))

print(
    "Teacher comparisons:",
    int(pair_candidates["requires_teacher"].sum()),
    "pairs x 2 swapped judgments"
)

Metric prefilter:   0%|          | 0/12250 [00:00<?, ?it/s]

metric_label,-1,0,1,All
config,,,,
EG,701,403,615,1719
JO,731,347,604,1682
LB,629,334,581,1544
MA,730,527,647,1904
MR,638,428,619,1685
OM,730,399,646,1775
PS,685,363,578,1626
SA,683,391,600,1674
SY,673,340,622,1635


Teacher comparisons: 14241 pairs x 2 swapped judgments


In [27]:
def context_string(value):
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except Exception:
            value = []

    if not isinstance(value, list):
        return "None"

    parts = []

    for turn in value:
        if not isinstance(turn, dict):
            continue

        speaker = str(turn.get("speaker", "")).strip()
        direction = str(turn.get("direction", "")).strip()
        text = str(turn.get("text", "")).strip()
        prefix = "/".join(
            item for item in [speaker, direction] if item
        )

        if text:
            parts.append(f"{prefix}: {text}" if prefix else text)

    return " <turn> ".join(parts) if parts else "None"

def reranker_query(row):
    code = str(row["config"])
    dialect = DIALECT_NAMES.get(code, code)

    return (
        f"Domain: {row.get('domain', '')}\n"
        f"Speaker: {row.get('speaker', '')}\n"
        f"Direction: {row.get('gender_direction', '')}\n"
        f"Previous English turns: "
        f"{context_string(row.get('previous_english_turns', []))}\n"
        f"Current English source: {row['source_text']}\n"
        f"Requested dialect: {code} ({dialect})"
    )

TEACHER_SYSTEM = (
    "You are a strict multilingual machine-translation evaluator. "
    "Follow the rubric and return only WINNER: A, WINNER: B, "
    "or WINNER: TIE."
)

def teacher_prompt(pair, swapped=False):
    row_index = int(pair.row_idx)
    row = official_dev_df.iloc[row_index]
    baseline = str(candidate_texts[row_index, int(pair.baseline_idx)])
    alternative = str(candidate_texts[row_index, int(pair.alt_idx)])

    candidate_a, candidate_b = (
        (alternative, baseline)
        if swapped
        else (baseline, alternative)
    )

    return (
        "Compare two Arabic translations of the same English turn.\n"
        "Decision order: (1) preserve all meaning with no additions, "
        "omissions, polarity/entity/number errors; "
        "(2) respect conversation context, speaker and gender; "
        "(3) sound authentic in the requested dialect; "
        "(4) fluency. The reference is evidence, not the only valid wording. "
        "If both are equally valid or the difference is uncertain, choose TIE.\n\n"
        f"{reranker_query(row)}\n"
        f"Human reference in the requested dialect: {row['reference_arabic']}\n"
        f"Candidate A: {candidate_a}\n"
        f"Candidate B: {candidate_b}\n\n"
        "WINNER:"
    )

def parse_teacher_answer(text):
    upper = str(text).strip().upper()

    match = re.search(
        r"WINNER\s*[:\-]?\s*(?:CANDIDATE\s*)?(TIE|A|B)\b",
        upper
    )

    if not match:
        match = re.search(
            r"\[?RESULT\]?\s*[:\-]?\s*(?:CANDIDATE\s*)?(TIE|A|B)\b",
            upper
        )

    if not match:
        match = re.match(r"\s*(TIE|A|B)\b", upper)

    return match.group(1) if match else "INVALID"

def answer_to_label(answer, swapped):
    if answer == "TIE":
        return 0

    if answer not in {"A", "B"}:
        return 99

    if not swapped:
        return -1 if answer == "A" else 1

    return 1 if answer == "A" else -1

In [29]:
# ============================================================
# CELL 7 — REASONED, ORDER-SWAPPED M-PROMETHEUS LABELING
# Resumable; previous invalid v1/v2 caches are ignored.
# ============================================================

TEACHER_CACHE_PATH = (
    RUN_DIR
    / "teacher_pair_judgments_reasoned_v4.csv"
)

TEACHER_REASONING_TOKENS = 96

print("Teacher cache:", TEACHER_CACHE_PATH)
print("Loading M-Prometheus from:", TEACHER_PATH)

teacher_tokenizer = AutoTokenizer.from_pretrained(
    TEACHER_PATH,
    use_fast=True,
    padding_side="left"
)

teacher_tokenizer.truncation_side = "left"

if teacher_tokenizer.pad_token_id is None:
    teacher_tokenizer.pad_token = (
        teacher_tokenizer.eos_token
    )

teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_PATH,
    dtype=AMP_DTYPE,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa"
).to(DEVICE).eval()

teacher_model.generation_config.temperature = None
teacher_model.generation_config.top_p = None
teacher_model.generation_config.top_k = None

print(
    "Teacher loaded on:",
    next(teacher_model.parameters()).device
)

print(
    "Allocated GPU memory:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

# ------------------------------------------------------------
# Force the final decision to be one valid token:
# 1 = Candidate A
# 2 = Candidate B
# 3 = Tie
# ------------------------------------------------------------

decision_codes = {
    "1": "A",
    "2": "B",
    "3": "TIE"
}

decision_token_to_answer = {}

for digit, answer in decision_codes.items():
    token_ids = teacher_tokenizer.encode(
        digit,
        add_special_tokens=False
    )

    if len(token_ids) != 1:
        raise RuntimeError(
            f"Decision code {digit!r} is not "
            "a single tokenizer token."
        )

    decision_token_to_answer[
        token_ids[0]
    ] = answer

allowed_decision_tokens = list(
    decision_token_to_answer
)

DECISION_REQUEST = (
    "Based only on your evaluation above, "
    "return exactly one digit:\n"
    "1 = Candidate A is better\n"
    "2 = Candidate B is better\n"
    "3 = Both are equally valid or uncertain\n"
    "Return one digit only."
)

REASONING_SYSTEM = (
    "You are a strict multilingual "
    "machine-translation evaluator. "
    "Analyze meaning preservation, omissions, "
    "additions, entities, numbers, polarity, "
    "conversation context, speaker, gender, "
    "fluency and requested-dialect authenticity."
)

def decision_tokens_allowed(
    batch_id,
    input_ids
):
    return allowed_decision_tokens

def make_reasoning_messages(
    pair,
    swapped
):
    return [
        {
            "role": "system",
            "content": REASONING_SYSTEM
        },
        {
            "role": "user",
            "content": teacher_prompt(
                pair,
                swapped
            )
        }
    ]

def make_decision_messages(
    pair,
    swapped,
    reasoning
):
    return [
        {
            "role": "system",
            "content": REASONING_SYSTEM
        },
        {
            "role": "user",
            "content": teacher_prompt(
                pair,
                swapped
            )
        },
        {
            "role": "assistant",
            "content": reasoning
        },
        {
            "role": "user",
            "content": DECISION_REQUEST
        }
    ]

# ------------------------------------------------------------
# Resume from the new valid cache only.
# ------------------------------------------------------------

if TEACHER_CACHE_PATH.exists():
    existing_teacher = pd.read_csv(
        TEACHER_CACHE_PATH
    )

    existing_teacher = (
        existing_teacher
        .drop_duplicates(
            "pair_id",
            keep="last"
        )
    )
else:
    existing_teacher = pd.DataFrame()

done_ids = (
    set(
        existing_teacher[
            "pair_id"
        ].astype(str)
    )
    if len(existing_teacher)
    else set()
)

pending_teacher = pair_candidates[
    pair_candidates["requires_teacher"]
    & ~pair_candidates[
        "pair_id"
    ].astype(str).isin(done_ids)
]

pending_records = list(
    pending_teacher.itertuples(
        index=False
    )
)

total_teacher_pairs = int(
    pair_candidates[
        "requires_teacher"
    ].sum()
)

print(
    "Valid judgments already saved:",
    len(done_ids),
    "/",
    total_teacher_pairs
)

print(
    "Pending teacher pairs:",
    len(pending_records)
)

if pending_records:
    print(
        "Each batch contains",
        TEACHER_PAIR_BATCH_SIZE,
        "pairs and",
        2 * TEACHER_PAIR_BATCH_SIZE,
        "order-swapped evaluations."
    )

# ------------------------------------------------------------
# Run two-stage evaluation:
# Stage 1: generate reasoning.
# Stage 2: force one decision token.
# ------------------------------------------------------------

progress = tqdm(
    range(
        0,
        len(pending_records),
        TEACHER_PAIR_BATCH_SIZE
    ),
    desc="M-Prometheus reasoned swapped judgments",
    dynamic_ncols=True,
    mininterval=0.5
)

for start in progress:
    batch_started = time.time()

    records = pending_records[
        start:
        start + TEACHER_PAIR_BATCH_SIZE
    ]

    reasoning_prompts = []

    for pair in records:
        for swapped in [False, True]:
            messages = make_reasoning_messages(
                pair,
                swapped
            )

            reasoning_prompts.append(
                teacher_tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
            )

    if start == 0:
        print(
            "First batch: generating teacher reasoning..."
        )

    reasoning_inputs = teacher_tokenizer(
        reasoning_prompts,
        padding=True,
        truncation=True,
        max_length=TEACHER_MAX_LENGTH,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.inference_mode():
        reasoning_generated = (
            teacher_model.generate(
                **reasoning_inputs,
                do_sample=False,
                max_new_tokens=(
                    TEACHER_REASONING_TOKENS
                ),
                pad_token_id=(
                    teacher_tokenizer
                    .pad_token_id
                ),
                eos_token_id=(
                    teacher_tokenizer
                    .eos_token_id
                )
            )
        )

    prompt_length = reasoning_inputs[
        "input_ids"
    ].shape[1]

    reasoning_outputs = (
        teacher_tokenizer.batch_decode(
            reasoning_generated[
                :,
                prompt_length:
            ],
            skip_special_tokens=True
        )
    )

    del reasoning_inputs
    del reasoning_generated

    decision_prompts = []
    reasoning_offset = 0

    for pair in records:
        for swapped in [False, True]:
            reasoning = reasoning_outputs[
                reasoning_offset
            ]

            messages = make_decision_messages(
                pair,
                swapped,
                reasoning
            )

            decision_prompts.append(
                teacher_tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
            )

            reasoning_offset += 1

    if start == 0:
        print(
            "First batch: producing constrained decisions..."
        )

    decision_inputs = teacher_tokenizer(
        decision_prompts,
        padding=True,
        truncation=True,
        max_length=TEACHER_MAX_LENGTH,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.inference_mode():
        decision_generated = (
            teacher_model.generate(
                **decision_inputs,
                do_sample=False,
                max_new_tokens=1,
                prefix_allowed_tokens_fn=(
                    decision_tokens_allowed
                ),
                pad_token_id=(
                    teacher_tokenizer
                    .pad_token_id
                ),
                eos_token_id=(
                    teacher_tokenizer
                    .eos_token_id
                )
            )
        )

    generated_decision_ids = (
        decision_generated[:, -1]
        .detach()
        .cpu()
        .tolist()
    )

    decisions = [
        decision_token_to_answer.get(
            token_id,
            "INVALID"
        )
        for token_id
        in generated_decision_ids
    ]

    del decision_inputs
    del decision_generated

    output_rows = []

    for offset, pair in enumerate(records):
        forward_answer = decisions[
            2 * offset
        ]
        reverse_answer = decisions[
            2 * offset + 1
        ]

        forward_reasoning = (
            reasoning_outputs[
                2 * offset
            ]
        )
        reverse_reasoning = (
            reasoning_outputs[
                2 * offset + 1
            ]
        )

        output_rows.append({
            "pair_id": str(
                pair.pair_id
            ),
            "forward_answer": (
                forward_answer
            ),
            "reverse_answer": (
                reverse_answer
            ),
            "forward_label": (
                answer_to_label(
                    forward_answer,
                    False
                )
            ),
            "reverse_label": (
                answer_to_label(
                    reverse_answer,
                    True
                )
            ),
            "forward_output": (
                forward_reasoning
            ),
            "reverse_output": (
                reverse_reasoning
            )
        })

    batch_df = pd.DataFrame(
        output_rows
    )

    batch_invalid = (
        (
            batch_df[
                "forward_label"
            ] == 99
        )
        | (
            batch_df[
                "reverse_label"
            ] == 99
        )
    )

    if batch_invalid.any():
        display(batch_df)

        raise RuntimeError(
            "A constrained teacher decision "
            "was invalid."
        )

    file_already_exists = (
        TEACHER_CACHE_PATH.exists()
    )

    batch_df.to_csv(
        TEACHER_CACHE_PATH,
        mode="a",
        header=not file_already_exists,
        index=False,
        encoding="utf-8-sig"
    )

    completed_pairs = min(
        start + len(records),
        len(pending_records)
    )

    progress.set_postfix(
        completed=(
            len(done_ids)
            + completed_pairs
        ),
        batch_seconds=(
            f"{time.time() - batch_started:.1f}"
        ),
        gpu_gb=(
            f"{torch.cuda.memory_allocated() / 1024**3:.1f}"
        )
    )

    if start == 0:
        print(
            "First batch completed in",
            f"{time.time() - batch_started:.1f}",
            "seconds."
        )

        display(batch_df[[
            "pair_id",
            "forward_answer",
            "reverse_answer",
            "forward_output",
            "reverse_output"
        ]])

    del reasoning_outputs
    del batch_df

# ------------------------------------------------------------
# Release the teacher and validate the completed cache.
# ------------------------------------------------------------

del teacher_model
gc.collect()
torch.cuda.empty_cache()

if not TEACHER_CACHE_PATH.exists():
    raise RuntimeError(
        "Teacher cache was not created."
    )

teacher_judgments = (
    pd.read_csv(
        TEACHER_CACHE_PATH
    )
    .drop_duplicates(
        "pair_id",
        keep="last"
    )
)

expected_pair_ids = set(
    pair_candidates.loc[
        pair_candidates[
            "requires_teacher"
        ],
        "pair_id"
    ].astype(str)
)

actual_pair_ids = set(
    teacher_judgments[
        "pair_id"
    ].astype(str)
)

missing_pair_ids = (
    expected_pair_ids
    - actual_pair_ids
)

if missing_pair_ids:
    raise RuntimeError(
        f"Teacher cache is incomplete: "
        f"{len(missing_pair_ids):,} "
        "pairs are missing."
    )

invalid_rate = float((
    (
        teacher_judgments[
            "forward_label"
        ] == 99
    )
    | (
        teacher_judgments[
            "reverse_label"
        ] == 99
    )
).mean())

agreement_rate = float((
    teacher_judgments[
        "forward_label"
    ]
    == teacher_judgments[
        "reverse_label"
    ]
).mean())

print(
    "Completed teacher judgments:",
    len(teacher_judgments)
)

print(
    f"Teacher invalid rate: "
    f"{invalid_rate:.2%}"
)

print(
    f"Order-corrected teacher agreement: "
    f"{agreement_rate:.2%}"
)

display(pd.crosstab(
    teacher_judgments[
        "forward_label"
    ],
    teacher_judgments[
        "reverse_label"
    ],
    margins=True
))

if invalid_rate > 0:
    raise RuntimeError(
        "Teacher cache contains invalid "
        "constrained decisions."
    )

Teacher cache: /home/mabdallah/alexandriax_mt_14d/rerankers/95_qwen3_06b_selective_uplift_cleanlabels_v1/teacher_pair_judgments_reasoned_v4.csv
Loading M-Prometheus from: /home/mabdallah/alexandriax_mt_14d/models/hf/M-Prometheus-3B


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Teacher loaded on: cuda:0
Allocated GPU memory: 5.86 GB
Valid judgments already saved: 14241 / 14241
Pending teacher pairs: 0


M-Prometheus reasoned swapped judgments: 0it [00:00, ?it/s]

Completed teacher judgments: 14241
Teacher invalid rate: 0.00%
Order-corrected teacher agreement: 32.63%


reverse_label,-1,0,1,All
forward_label,,,,
-1,2439,113,7793,10345
0,73,72,117,262
1,1438,60,2136,3634
All,3950,245,10046,14241


In [30]:
CLEAN_PAIR_PATH = RUN_DIR / "clean_threeway_pairs.csv"

non_ties = pair_candidates[
    pair_candidates["metric_label"] != 0
].merge(
    teacher_judgments,
    on="pair_id",
    how="left",
    validate="one_to_one"
)

accepted_non_ties = non_ties[
    (non_ties["forward_label"] == non_ties["metric_label"])
    & (non_ties["reverse_label"] == non_ties["metric_label"])
].copy()

accepted_non_ties["label"] = (
    accepted_non_ties["metric_label"].astype(int)
)
accepted_non_ties["label_source"] = (
    "metric_and_swapped_teacher_agreement"
)

accepted_ties = pair_candidates[
    pair_candidates["metric_label"] == 0
].copy()

accepted_ties["label"] = 0
accepted_ties["label_source"] = "both_metrics_near_tie"

clean_pairs = pd.concat(
    [accepted_non_ties, accepted_ties],
    ignore_index=True,
    sort=False
)

clean_pairs = (
    clean_pairs
    .sort_values(["row_idx", "label"])
    .drop_duplicates("pair_id")
    .reset_index(drop=True)
)

clean_pairs["weight"] = np.where(
    clean_pairs["label"] == 0,
    0.75,
    1.0
).astype(np.float32)

clean_pairs.to_csv(
    CLEAN_PAIR_PATH,
    index=False,
    encoding="utf-8-sig"
)

acceptance = (
    len(accepted_non_ties)
    / max(1, len(non_ties))
)

print(
    f"Teacher-confirmed non-tie acceptance: "
    f"{acceptance:.2%}"
)

display(pd.crosstab(
    clean_pairs["config"],
    clean_pairs["label"],
    margins=True
))

if (
    len(clean_pairs) < 2000
    or clean_pairs["label"].nunique() < 3
):
    raise RuntimeError(
        "Too few clean three-way labels; "
        "inspect teacher agreement and metric thresholds."
    )

Teacher-confirmed non-tie acceptance: 17.54%


label,-1,0,1,All
config,,,,
EG,116,403,86,605
JO,138,347,104,589
LB,103,334,90,527
MA,137,527,110,774
MR,167,428,116,711
OM,121,399,98,618
PS,103,363,92,558
SA,117,391,91,599
SY,117,340,82,539


In [31]:
RERANK_INSTRUCTION = (
    "Given an English source turn, its conversation context, and a requested "
    "Arabic dialect, judge whether the Arabic document is a faithful, complete, "
    "context-consistent, fluent translation in that exact dialect. "
    "Meaning correctness dominates dialect style; reject additions, omissions "
    "and wrong-dialect wording."
)

QWEN_PREFIX = (
    '<|im_start|>system\n Judge whether the Document meets the requirements '
    'based on the Query and the Instruct provided. Note that the answer can '
    'only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
)

QWEN_SUFFIX = (
    "<|im_end|>\n<|im_start|>assistant\n"
    "<think>\n\n</think>\n\n"
)

rank_tokenizer = AutoTokenizer.from_pretrained(
    RERANKER_PATH,
    use_fast=True,
    padding_side="left"
)
rank_tokenizer.truncation_side = "left"

if rank_tokenizer.pad_token_id is None:
    rank_tokenizer.pad_token = rank_tokenizer.eos_token

prefix_tokens = rank_tokenizer.encode(
    QWEN_PREFIX,
    add_special_tokens=False
)
suffix_tokens = rank_tokenizer.encode(
    QWEN_SUFFIX,
    add_special_tokens=False
)

token_no_id = rank_tokenizer.convert_tokens_to_ids("no")
token_yes_id = rank_tokenizer.convert_tokens_to_ids("yes")

def format_rank_document(row_index, candidate_index):
    query = reranker_query(
        official_dev_df.iloc[int(row_index)]
    )
    document = str(
        candidate_texts[
            int(row_index),
            int(candidate_index)
        ]
    )

    return (
        f"<Instruct>: {RERANK_INSTRUCTION}\n"
        f"<Query>: {query}\n"
        f"<Document>: {document}"
    )

def tokenize_rank_documents(
    row_indices,
    candidate_indices
):
    texts = [
        format_rank_document(row, candidate)
        for row, candidate
        in zip(row_indices, candidate_indices)
    ]

    encoded = rank_tokenizer(
        texts,
        padding=False,
        truncation="longest_first",
        return_attention_mask=False,
        max_length=(
            RANK_MAX_LENGTH
            - len(prefix_tokens)
            - len(suffix_tokens)
        )
    )

    encoded["input_ids"] = [
        prefix_tokens + ids + suffix_tokens
        for ids in encoded["input_ids"]
    ]

    return rank_tokenizer.pad(
        encoded,
        padding=True,
        return_tensors="pt",
        max_length=RANK_MAX_LENGTH
    )

class CleanPairDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]

        return (
            int(row.row_idx),
            int(row.alt_idx),
            int(row.label),
            float(row.weight)
        )

class CleanPairCollator:
    def __call__(self, items):
        rows, alternatives, labels, weights = zip(*items)

        baselines = [
            int(baseline_idx[row])
            for row in rows
        ]

        inputs = tokenize_rank_documents(
            list(rows) + list(rows),
            list(alternatives) + baselines
        )

        inputs["labels_3way"] = torch.tensor(
            labels,
            dtype=torch.float32
        )
        inputs["pair_weights"] = torch.tensor(
            weights,
            dtype=torch.float32
        )
        inputs["pair_batch_size"] = len(items)

        return inputs

def create_rank_model():
    base = AutoModelForCausalLM.from_pretrained(
        RERANKER_PATH,
        torch_dtype=AMP_DTYPE,
        low_cpu_mem_usage=True,
        attn_implementation="sdpa"
    )

    base.config.use_cache = False
    base.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        }
    )

    lora = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        bias="none"
    )

    model = get_peft_model(base, lora).to(DEVICE)
    model.enable_input_require_grads()

    return model

def qwen_log_odds(model, inputs):
    outputs = model(
        **inputs,
        use_cache=False,
        logits_to_keep=1
    )

    last_logits = outputs.logits[:, -1, :]

    return (
        last_logits[:, token_yes_id].float()
        - last_logits[:, token_no_id].float()
    )

def adapter_state_cpu(model):
    return {
        key: value.detach().cpu()
        for key, value
        in get_peft_model_state_dict(model).items()
    }

In [32]:
def core_calibration_split(outer_fold):
    outer_train = np.where(fold_id != outer_fold)[0]
    core, calibration = [], []
    rng = np.random.default_rng(
        SEED + 10000 + outer_fold
    )
    configs = official_dev_df["config"].to_numpy()
    conversations = (
        official_dev_df["config"]
        + "::"
        + official_dev_df["conversation_id"]
    ).to_numpy()

    for country in sorted(set(configs[outer_train])):
        country_rows = outer_train[
            configs[outer_train] == country
        ]
        unique_groups = np.unique(
            conversations[country_rows]
        )
        rng.shuffle(unique_groups)

        n_cal = min(
            max(
                1,
                int(round(
                    len(unique_groups)
                    * CALIBRATION_FRACTION
                ))
            ),
            len(unique_groups) - 1
        )

        cal_groups = set(unique_groups[:n_cal])
        mask = np.asarray([
            group in cal_groups
            for group in conversations[country_rows]
        ])

        calibration.extend(
            country_rows[mask].tolist()
        )
        core.extend(
            country_rows[~mask].tolist()
        )

    core = np.asarray(
        sorted(core),
        dtype=np.int64
    )
    calibration = np.asarray(
        sorted(calibration),
        dtype=np.int64
    )

    if set(core) & set(calibration):
        raise RuntimeError(
            "Core/calibration leakage."
        )

    return core, calibration

def score_rows(model, row_indices, desc):
    row_indices = np.asarray(
        row_indices,
        dtype=np.int64
    )
    flat_rows = np.repeat(row_indices, K)
    flat_candidates = np.tile(
        np.arange(K, dtype=np.int64),
        len(row_indices)
    )
    flat_scores = np.empty(
        len(flat_rows),
        dtype=np.float32
    )

    model.eval()

    for start in tqdm(
        range(0, len(flat_rows), SCORE_BATCH_SIZE),
        desc=desc,
        leave=False
    ):
        end = min(
            start + SCORE_BATCH_SIZE,
            len(flat_rows)
        )

        batch = tokenize_rank_documents(
            flat_rows[start:end],
            flat_candidates[start:end]
        )
        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.inference_mode(), amp_context():
            flat_scores[start:end] = (
                qwen_log_odds(model, batch)
                .cpu()
                .numpy()
            )

    return flat_scores.reshape(
        len(row_indices),
        K
    )

def selections_at_threshold(
    scores,
    row_indices,
    threshold
):
    row_indices = np.asarray(
        row_indices,
        dtype=np.int64
    )
    baseline_local_idx = baseline_idx[row_indices]
    baseline_scores = scores[
        np.arange(len(row_indices)),
        baseline_local_idx
    ]
    margins = scores - baseline_scores[:, None]

    for local, row_index in enumerate(row_indices):
        base_text = normalized_text(
            candidate_texts[
                row_index,
                baseline_local_idx[local]
            ]
        )

        for candidate_index in range(K):
            if (
                candidate_index
                != baseline_local_idx[local]
                and normalized_text(
                    candidate_texts[
                        row_index,
                        candidate_index
                    ]
                ) == base_text
            ):
                margins[
                    local,
                    candidate_index
                ] = -np.inf

    best_idx = margins.argmax(1)
    best_margin = margins[
        np.arange(len(row_indices)),
        best_idx
    ]
    selected_idx = np.where(
        best_margin > threshold,
        best_idx,
        baseline_local_idx
    )

    return (
        selected_idx.astype(np.int16),
        best_margin.astype(np.float32)
    )

def calibrate_threshold(
    scores,
    row_indices,
    fold,
    epoch
):
    baseline_local = baseline_idx[row_indices]
    positive_margins = []

    raw_margins = (
        scores
        - scores[
            np.arange(len(row_indices)),
            baseline_local
        ][:, None]
    )

    for local, row_index in enumerate(row_indices):
        for candidate_index in range(K):
            if candidate_index == baseline_local[local]:
                continue

            if normalized_text(
                candidate_texts[
                    row_index,
                    candidate_index
                ]
            ) == normalized_text(
                candidate_texts[
                    row_index,
                    baseline_local[local]
                ]
            ):
                continue

            if raw_margins[
                local,
                candidate_index
            ] > 0:
                positive_margins.append(float(
                    raw_margins[
                        local,
                        candidate_index
                    ]
                ))

    thresholds = [NO_OVERRIDE_THRESHOLD]

    if positive_margins:
        thresholds += np.unique(np.quantile(
            positive_margins,
            np.linspace(
                0,
                1,
                CALIBRATION_THRESHOLD_POINTS
            )
        )).tolist()

    base_predictions = baseline_predictions[
        row_indices
    ]

    base_summary, _, _ = (
        official_metrics_for_indices(
            base_predictions,
            row_indices,
            "calibration_baseline"
        )
    )

    rows = []

    for threshold in thresholds:
        selected, _ = selections_at_threshold(
            scores,
            row_indices,
            threshold
        )
        predictions = candidate_texts[
            row_indices,
            selected
        ]

        summary, _, _ = (
            official_metrics_for_indices(
                predictions,
                row_indices,
                "calibration_candidate"
            )
        )

        overrides = int(sum(
            normalized_text(a)
            != normalized_text(b)
            for a, b in zip(
                predictions,
                base_predictions
            )
        ))

        gain = (
            summary["Average spBLEU (primary)"]
            - base_summary[
                "Average spBLEU (primary)"
            ]
        )

        if (
            threshold < NO_OVERRIDE_THRESHOLD
            and overrides
            < CALIBRATION_MIN_OVERRIDES
        ):
            continue

        rows.append({
            "threshold": float(threshold),
            "gain": float(gain),
            "overrides": overrides,
            "spBLEU": summary[
                "Average spBLEU (primary)"
            ]
        })

    curve = (
        pd.DataFrame(rows)
        .sort_values(
            ["gain", "threshold", "overrides"],
            ascending=[False, False, True]
        )
        .reset_index(drop=True)
    )

    curve.to_csv(
        RUN_DIR
        / f"fold_{fold}"
        / f"calibration_epoch_{epoch}.csv",
        index=False
    )

    best = curve.iloc[0]

    if best["gain"] <= 0:
        return NO_OVERRIDE_THRESHOLD, 0.0, 0

    return (
        float(best["threshold"]),
        float(best["gain"]),
        int(best["overrides"])
    )

def train_fold(
    outer_fold,
    core_indices,
    calibration_indices
):
    fold_dir = RUN_DIR / f"fold_{outer_fold}"
    fold_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    last_path = (
        fold_dir
        / "checkpoint_last.pt"
    )
    best_path = (
        fold_dir
        / "best_adapter.pt"
    )

    train_pairs = clean_pairs[
        clean_pairs["row_idx"].isin(
            set(core_indices)
        )
    ].copy().reset_index(drop=True)

    counts = (
        train_pairs["label"]
        .value_counts()
        .to_dict()
    )

    class_weights = {
        label: float(np.clip(
            len(train_pairs)
            / (3 * count),
            0.5,
            3.0
        ))
        for label, count in counts.items()
    }

    train_pairs["weight"] *= (
        train_pairs["label"]
        .map(class_weights)
        .astype(np.float32)
    )

    print(
        f"Fold {outer_fold}: "
        f"core={len(core_indices):,}, "
        f"calibration={len(calibration_indices):,}, "
        f"clean_pairs={len(train_pairs):,}, "
        f"labels={counts}"
    )

    model = create_rank_model()

    optimizer = torch.optim.AdamW(
        [
            parameter
            for parameter in model.parameters()
            if parameter.requires_grad
        ],
        lr=RANK_LR,
        weight_decay=RANK_WEIGHT_DECAY
    )

    batches_per_epoch = math.ceil(
        len(train_pairs)
        / RANK_BATCH_SIZE
    )
    updates_per_epoch = math.ceil(
        batches_per_epoch
        / RANK_GRAD_ACCUM_STEPS
    )
    total_updates = (
        updates_per_epoch
        * RANK_EPOCHS
    )

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        int(
            total_updates
            * RANK_WARMUP_RATIO
        ),
        total_updates
    )

    scaler = new_scaler()
    start_epoch = 0
    start_batch = 0
    global_step = 0
    best_gain = -float("inf")
    best_threshold = NO_OVERRIDE_THRESHOLD
    best_epoch = -1

    if last_path.exists():
        state = load_torch(last_path)

        if state["signature"] != RUN_SIGNATURE:
            raise RuntimeError(
                f"Fold {outer_fold} checkpoint "
                "signature mismatch; change RUN_NAME."
            )

        set_peft_model_state_dict(
            model,
            state["adapter"]
        )
        optimizer.load_state_dict(
            state["optimizer"]
        )
        optimizer_to(
            optimizer,
            DEVICE
        )
        scheduler.load_state_dict(
            state["scheduler"]
        )
        scaler.load_state_dict(
            state["scaler"]
        )

        start_epoch = state["epoch"]
        start_batch = state["next_batch"]
        global_step = state["step"]
        best_gain = state["best_gain"]
        best_threshold = state["best_threshold"]
        best_epoch = state["best_epoch"]
        restore_rng(state["rng"])

        print(
            f"Fold {outer_fold}: "
            f"resume epoch={start_epoch}, "
            f"batch={start_batch}, "
            f"step={global_step}"
        )

    dataset = CleanPairDataset(
        train_pairs
    )

    for epoch in range(
        start_epoch,
        RANK_EPOCHS
    ):
        generator = torch.Generator().manual_seed(
            SEED
            + 1000 * outer_fold
            + epoch
        )

        loader = DataLoader(
            dataset,
            batch_size=RANK_BATCH_SIZE,
            shuffle=True,
            generator=generator,
            collate_fn=CleanPairCollator(),
            num_workers=NUM_WORKERS,
            pin_memory=True
        )

        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        progress = tqdm(
            enumerate(loader),
            total=len(loader),
            desc=(
                f"Fold {outer_fold + 1}/{N_FOLDS} "
                f"epoch {epoch + 1}/{RANK_EPOCHS}"
            )
        )

        for batch_index, batch in progress:
            if (
                epoch == start_epoch
                and batch_index < start_batch
            ):
                continue

            labels = batch.pop(
                "labels_3way"
            ).to(DEVICE)
            weights = batch.pop(
                "pair_weights"
            ).to(DEVICE)
            pair_batch_size = batch.pop(
                "pair_batch_size"
            )

            inputs = {
                key: value.to(
                    DEVICE,
                    non_blocking=True
                )
                for key, value in batch.items()
            }

            with amp_context():
                scores = qwen_log_odds(
                    model,
                    inputs
                )

                difference = (
                    scores[:pair_batch_size]
                    - scores[pair_batch_size:]
                )

                non_tie = labels != 0
                losses = torch.empty_like(
                    difference
                )

                losses[non_tie] = F.softplus(
                    -(
                        labels[non_tie]
                        * difference[non_tie]
                    )
                    / RANK_TEMPERATURE
                )

                losses[~non_tie] = F.relu(
                    difference[~non_tie].abs()
                    - TIE_SCORE_MARGIN
                ).pow(2)

                loss = (
                    losses
                    * weights
                ).sum() / weights.sum()

            scaler.scale(
                loss
                / RANK_GRAD_ACCUM_STEPS
            ).backward()

            running_loss += float(
                loss.detach()
            )

            do_step = (
                (batch_index + 1)
                % RANK_GRAD_ACCUM_STEPS
                == 0
                or batch_index + 1
                == len(loader)
            )

            if do_step:
                scaler.unscale_(
                    optimizer
                )
                nn.utils.clip_grad_norm_(
                    model.parameters(),
                    RANK_MAX_GRAD_NORM
                )

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(
                    set_to_none=True
                )
                global_step += 1

                progress.set_postfix(
                    loss=(
                        f"{running_loss / max(1, batch_index + 1):.4f}"
                    ),
                    lr=(
                        f"{scheduler.get_last_lr()[0]:.2e}"
                    ),
                    step=global_step
                )

                if (
                    global_step
                    % RANK_SAVE_STEPS
                    == 0
                ):
                    atomic_torch_save({
                        "signature": RUN_SIGNATURE,
                        "adapter": adapter_state_cpu(model),
                        "optimizer": optimizer.state_dict(),
                        "scheduler": scheduler.state_dict(),
                        "scaler": scaler.state_dict(),
                        "epoch": epoch,
                        "next_batch": batch_index + 1,
                        "step": global_step,
                        "best_gain": best_gain,
                        "best_threshold": best_threshold,
                        "best_epoch": best_epoch,
                        "rng": capture_rng()
                    }, last_path)

        calibration_scores = score_rows(
            model,
            calibration_indices,
            (
                f"Fold {outer_fold + 1} "
                f"calibration epoch {epoch + 1}"
            )
        )

        threshold, gain, overrides = (
            calibrate_threshold(
                calibration_scores,
                calibration_indices,
                outer_fold,
                epoch + 1
            )
        )

        print(
            f"Fold {outer_fold + 1} "
            f"epoch {epoch + 1}: "
            f"calibration gain={gain:+.4f}, "
            f"threshold={threshold:.5f}, "
            f"overrides={overrides}"
        )

        if (
            gain > best_gain + 1e-12
            or (
                abs(gain - best_gain) <= 1e-12
                and threshold > best_threshold
            )
        ):
            best_gain = gain
            best_threshold = threshold
            best_epoch = epoch + 1

            atomic_torch_save({
                "signature": RUN_SIGNATURE,
                "adapter": adapter_state_cpu(model),
                "threshold": best_threshold,
                "gain": best_gain,
                "epoch": best_epoch
            }, best_path)

        start_batch = 0

        atomic_torch_save({
            "signature": RUN_SIGNATURE,
            "adapter": adapter_state_cpu(model),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "epoch": epoch + 1,
            "next_batch": 0,
            "step": global_step,
            "best_gain": best_gain,
            "best_threshold": best_threshold,
            "best_epoch": best_epoch,
            "rng": capture_rng()
        }, last_path)

    best_state = load_torch(
        best_path
    )

    set_peft_model_state_dict(
        model,
        best_state["adapter"]
    )

    model.eval()
    model.save_pretrained(
        fold_dir
        / "best_adapter"
    )
    rank_tokenizer.save_pretrained(
        fold_dir
        / "best_adapter"
    )

    return (
        model,
        float(best_state["threshold"]),
        {
            "best_epoch": int(
                best_state["epoch"]
            ),
            "calibration_gain": float(
                best_state["gain"]
            )
        }
    )

In [36]:
# Fold Reconstruction Cell
from sklearn.model_selection import StratifiedGroupKFold

groups = (
    official_dev_df["config"].astype(str)
    + "::"
    + official_dev_df[
        "conversation_id"
    ].astype(str)
)

fold_id = np.full(
    N,
    -1,
    dtype=np.int8
)

splitter = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

for fold, (
    training_indices,
    validation_indices
) in enumerate(
    splitter.split(
        np.zeros(N),
        official_dev_df["config"],
        groups
    )
):
    fold_id[
        validation_indices
    ] = fold

if (fold_id < 0).any():
    raise RuntimeError(
        "Fold assignment is incomplete."
    )

fold_check = (
    official_dev_df
    .assign(fold=fold_id)
    .groupby([
        "config",
        "conversation_id"
    ])["fold"]
    .nunique()
    .max()
)

if fold_check != 1:
    raise RuntimeError(
        "Conversation leakage across folds."
    )

fold_counts = pd.crosstab(
    official_dev_df["config"],
    fold_id,
    margins=True
)

display(fold_counts)

print(
    "Fold assignment ready:",
    np.bincount(
        fold_id,
        minlength=N_FOLDS
    ).tolist()
)

print(
    "Total assigned rows:",
    int(
        np.bincount(
            fold_id,
            minlength=N_FOLDS
        ).sum()
    )
)

col_0,0,1,2,3,4,All
config,,,,,,
EG,224,223,222,222,222,1113
JO,223,224,222,222,222,1113
LB,223,223,223,225,224,1118
MA,223,221,222,222,222,1110
MR,222,222,224,224,222,1114
OM,222,221,221,222,223,1109
PS,222,223,221,221,223,1110
SA,222,221,221,223,223,1110
SY,223,225,225,223,223,1119


Fold assignment ready: [2451, 2451, 2450, 2449, 2449]
Total assigned rows: 12250


In [40]:
oof_scores = np.full(
    (N, K),
    np.nan,
    dtype=np.float32
)
oof_selected_idx = baseline_idx.copy()
oof_best_margin = np.zeros(
    N,
    dtype=np.float32
)
oof_threshold = np.full(
    N,
    NO_OVERRIDE_THRESHOLD,
    dtype=np.float32
)
fold_metadata = []

for outer_fold in range(N_FOLDS):
    print("\n" + "=" * 90)
    print(
        f"OUTER FOLD "
        f"{outer_fold + 1}/{N_FOLDS}"
    )
    print("=" * 90)

    fold_dir = (
        RUN_DIR
        / f"fold_{outer_fold}"
    )
    fold_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    result_path = (
        fold_dir
        / "heldout_result.npz"
    )
    metadata_path = (
        fold_dir
        / "heldout_result.json"
    )

    heldout_indices = np.where(
        fold_id == outer_fold
    )[0]

    if (
        result_path.exists()
        and metadata_path.exists()
    ):
        metadata = json.loads(
            metadata_path.read_text(
                encoding="utf-8"
            )
        )
        cached = np.load(
            result_path
        )

        if (
            metadata["signature"]
            != RUN_SIGNATURE
            or not np.array_equal(
                cached["row_indices"],
                heldout_indices
            )
        ):
            raise RuntimeError(
                f"Fold {outer_fold} held-out "
                "cache mismatch; change RUN_NAME."
            )

        heldout_scores = cached["scores"]
        selected_idx = cached["selected_idx"]
        best_margin = cached["best_margin"]
        threshold = float(
            metadata["threshold"]
        )

    else:
        core_indices, calibration_indices = (
            core_calibration_split(
                outer_fold
            )
        )

        model, threshold, metadata = train_fold(
            outer_fold,
            core_indices,
            calibration_indices
        )

        heldout_scores = score_rows(
            model,
            heldout_indices,
            f"Fold {outer_fold + 1} held-out"
        )

        selected_idx, best_margin = (
            selections_at_threshold(
                heldout_scores,
                heldout_indices,
                threshold
            )
        )

        atomic_npz_save(
            result_path,
            row_indices=heldout_indices,
            scores=heldout_scores,
            selected_idx=selected_idx,
            best_margin=best_margin
        )

        atomic_json_save({
            "signature": RUN_SIGNATURE,
            "threshold": threshold,
            **metadata
        }, metadata_path)

        del model
        gc.collect()
        torch.cuda.empty_cache()

    oof_scores[
        heldout_indices
    ] = heldout_scores

    oof_selected_idx[
        heldout_indices
    ] = selected_idx

    oof_best_margin[
        heldout_indices
    ] = best_margin

    oof_threshold[
        heldout_indices
    ] = threshold

    fold_metadata.append({
        "fold": outer_fold,
        "threshold": threshold,
        **metadata
    })

if not np.isfinite(oof_scores).all():
    raise RuntimeError(
        "OOF score matrix is incomplete."
    )

atomic_numpy_save(
    oof_scores,
    OUTPUT_DIR
    / "oof_candidate_scores.npy"
)

pd.DataFrame(
    fold_metadata
).to_csv(
    OUTPUT_DIR
    / "fold_calibration_summary.csv",
    index=False
)


OUTER FOLD 1/5

OUTER FOLD 2/5

OUTER FOLD 3/5

OUTER FOLD 4/5

OUTER FOLD 5/5


In [41]:
selected_predictions = candidate_texts[
    np.arange(N),
    oof_selected_idx
]

selected_variants = np.asarray(
    variant_names,
    dtype=object
)[oof_selected_idx]

baseline_variants = np.asarray(
    variant_names,
    dtype=object
)[baseline_idx]

reranker_turn_df = official_dev_df[[
    "source_id",
    "config",
    "country",
    "conversation_id",
    "turn_order",
    "source_text"
]].copy()

reranker_turn_df["prediction"] = (
    selected_predictions
)
reranker_turn_df[
    "selected_from_variant"
] = selected_variants
reranker_turn_df[
    "selected_candidate_index"
] = oof_selected_idx
reranker_turn_df[
    "selected_score"
] = oof_scores[
    np.arange(N),
    oof_selected_idx
]
reranker_turn_df[
    "system92_score"
] = oof_scores[
    np.arange(N),
    baseline_idx
]
reranker_turn_df[
    "uplift_margin"
] = oof_best_margin
reranker_turn_df[
    "fold_threshold"
] = oof_threshold
reranker_turn_df[
    "baseline_from_variant"
] = baseline_variants
reranker_turn_df[
    "baseline_candidate_index"
] = baseline_idx

reranker_turn_df[
    "changed_from_system92"
] = [
    normalized_text(selected)
    != normalized_text(baseline)
    for selected, baseline
    in zip(
        selected_predictions,
        baseline_predictions
    )
]

reranker_turn_df.to_csv(
    OUTPUT_DIR
    / "reranker_oof_turn_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

reranker_summary, reranker_per_country, reranker_scored = (
    official_metrics_for_indices(
        selected_predictions,
        np.arange(N),
        RUN_NAME + "_OOF"
    )
)

baseline_summary, baseline_per_country, baseline_scored = (
    official_metrics_for_indices(
        baseline_predictions,
        np.arange(N),
        BASELINE_VARIANT
    )
)

comparison = pd.DataFrame([
    baseline_summary,
    reranker_summary
])

country_comparison = baseline_per_country.merge(
    reranker_per_country,
    on=["country", "turns"],
    suffixes=(
        "_system92",
        "_reranker"
    )
)

country_comparison[
    "spBLEU_delta"
] = (
    country_comparison[
        "spBLEU_reranker"
    ]
    - country_comparison[
        "spBLEU_system92"
    ]
)

country_comparison[
    "chrF++_delta"
] = (
    country_comparison[
        "chrF++_reranker"
    ]
    - country_comparison[
        "chrF++_system92"
    ]
)

display(comparison)
display(country_comparison)

reranker_gain = (
    reranker_summary[
        "Average spBLEU (primary)"
    ]
    - baseline_summary[
        "Average spBLEU (primary)"
    ]
)

deploy_reranker = (
    reranker_gain
    > DEPLOY_MIN_SPBLEU_GAIN
)

final_system = (
    RUN_NAME + "_OOF"
    if deploy_reranker
    else BASELINE_VARIANT
)

final_predictions = (
    selected_predictions
    if deploy_reranker
    else baseline_predictions
)

final_summary, final_per_country, final_scored = (
    official_metrics_for_indices(
        final_predictions,
        np.arange(N),
        final_system
    )
)

print(
    f"Text-level overrides: "
    f"{int(reranker_turn_df['changed_from_system92'].sum()):,}"
    f"/{N:,}"
)

print(
    f"OOF spBLEU gain: "
    f"{reranker_gain:+.6f} | "
    f"deployment: {final_system}"
)

comparison.to_csv(
    OUTPUT_DIR
    / "dev_system_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

country_comparison.to_csv(
    OUTPUT_DIR
    / "dev_per_country_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

final_per_country.to_csv(
    OUTPUT_DIR
    / "per_country_official_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "Variant": final_system,
    "Checkpoint": (
        "nested-calibrated 5-fold OOF"
    ),
    "Average spBLEU (primary)": (
        final_summary[
            "Average spBLEU (primary)"
        ]
    ),
    "Average chrF++": (
        final_summary[
            "Average chrF++"
        ]
    )
}]).to_csv(
    OUTPUT_DIR
    / "official_leaderboard_score_row.csv",
    index=False,
    encoding="utf-8-sig"
)

atomic_json_save({
    "reranker_gain_spBLEU": reranker_gain,
    "minimum_required_gain": (
        DEPLOY_MIN_SPBLEU_GAIN
    ),
    "deployed_system": final_system,
    "reranker_summary": reranker_summary,
    "baseline_summary": baseline_summary
}, OUTPUT_DIR / "deployment_decision.json")

submission_turn_df = (
    reranker_turn_df.copy()
)

if not deploy_reranker:
    submission_turn_df[
        "prediction"
    ] = baseline_predictions

    submission_turn_df[
        "selected_from_variant"
    ] = baseline_variants

    submission_turn_df[
        "selected_candidate_index"
    ] = baseline_idx

submission_turn_df.to_csv(
    OUTPUT_DIR
    / "turn_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

final_scored.to_csv(
    OUTPUT_DIR
    / "scored_turn_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

atomic_json_save({
    "variant_name": final_system,
    "evaluation": (
        "nested-calibrated "
        "conversation-grouped 5-fold OOF"
    ),
    "num_turns": N,
    "num_countries": (
        EXPECTED_DEV_COUNTRIES
    ),
    "Average spBLEU (primary)": (
        final_summary[
            "Average spBLEU (primary)"
        ]
    ),
    "Average chrF++": (
        final_summary[
            "Average chrF++"
        ]
    ),
    "reranker_spBLEU_gain_over_system92": (
        reranker_gain
    ),
    "oracle_gain": oracle_gain,
    "deployed_reranker": deploy_reranker,
    "training_epochs": RANK_EPOCHS,
    "candidate_variants": variant_names,
    "reranker": RERANKER_REPO,
    "teacher": TEACHER_REPO,
    "per_country": (
        final_per_country.to_dict(
            "records"
        )
    ),
    "fold_calibration": fold_metadata
}, OUTPUT_DIR / "official_metrics.json")

submission_records = []

ordered = final_scored.sort_values([
    "config",
    "conversation_id",
    "turn_order"
])

for (
    country,
    conversation_id
), conversation_df in ordered.groupby(
    ["config", "conversation_id"],
    sort=True
):
    if conversation_df[
        "turn_order"
    ].duplicated().any():
        raise RuntimeError(
            f"Duplicate turn order in "
            f"{country}/{conversation_id}"
        )

    turns = [
        {
            "turn_order": int(
                row.turn_order
            ),
            "prediction": str(
                row.prediction
            )
        }
        for row
        in conversation_df.itertuples()
    ]

    submission_records.append({
        "conv_id": str(
            conversation_id
        ),
        "country": str(country),
        "turns": turns
    })

jsonl_path = (
    OUTPUT_DIR
    / "predictions.jsonl"
)

with open(
    jsonl_path,
    "w",
    encoding="utf-8"
) as file:
    for record in submission_records:
        file.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )

zip_path = (
    OUTPUT_DIR
    / "submission_predictions.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:
    zip_file.write(
        jsonl_path,
        arcname="predictions.jsonl"
    )

readback_keys = set()
readback_turns = 0

with zipfile.ZipFile(
    zip_path
) as zip_file:
    if zip_file.namelist() != [
        "predictions.jsonl"
    ]:
        raise RuntimeError(
            "ZIP must contain only "
            "predictions.jsonl"
        )

    with zip_file.open(
        "predictions.jsonl"
    ) as file:
        for line in file:
            record = json.loads(
                line.decode("utf-8")
            )

            for turn in record["turns"]:
                readback_turns += 1

                readback_keys.add((
                    str(record["country"]),
                    str(record["conv_id"]),
                    int(turn["turn_order"])
                ))

expected_keys = set(zip(
    official_dev_df["config"],
    official_dev_df[
        "conversation_id"
    ],
    official_dev_df[
        "turn_order"
    ]
))

if (
    readback_turns
    != EXPECTED_DEV_TURNS
    or readback_keys
    != expected_keys
):
    raise RuntimeError(
        "Submission ZIP readback "
        "validation failed."
    )

print(
    "\nOFFICIAL-STYLE DEVELOPMENT RESULT"
)
print(
    "System:",
    final_system
)
print(
    "Average spBLEU:",
    f"{final_summary['Average spBLEU (primary)']:.6f}"
)
print(
    "Average chrF++:",
    f"{final_summary['Average chrF++']:.6f}"
)
print(
    "\nSUBMIT THIS ZIP:",
    zip_path
)

display(final_per_country)

,system,Average spBLEU (primary),Average chrF++
0,92_mixed_best_checkpoint_variant_per_country,30.928003,45.594526
1,95_qwen3_06b_selective_uplift_cleanlabels_v1_OOF,30.854149,45.580179


,country,turns,spBLEU_system92,chrF++_system92,BLEU_system92,chrF_system92,spBLEU_reranker,chrF++_reranker,BLEU_reranker,chrF_reranker,spBLEU_delta,chrF++_delta
0,EG,1113,32.903189,46.892198,19.510451,49.894903,32.544562,46.721045,19.345045,49.722044,-0.358627,-0.171153
1,JO,1113,35.301943,49.377895,19.819243,52.901916,35.209520,49.447347,19.767549,52.989322,-0.092422,0.069453
2,LB,1118,31.701457,45.871215,20.021014,48.586333,31.706727,45.851135,19.880188,48.580724,0.005270,-0.020080
3,MA,1110,23.530143,39.376601,12.865149,42.236961,23.401187,39.258305,12.843018,42.111340,-0.128956,-0.118296
4,MR,1114,17.380438,33.733013,7.100699,37.779153,17.438390,33.817957,7.139188,37.867208,0.057952,0.084944
5,OM,1109,36.147408,49.843410,20.805106,53.247687,36.092671,49.813716,20.726349,53.238727,-0.054737,-0.029694
6,PS,1110,33.418402,47.762506,20.699325,51.031341,33.529934,47.897889,20.796582,51.175909,0.111532,0.135383
7,SA,1110,33.166094,48.135685,18.903768,52.048449,33.226468,48.146706,18.915618,52.060176,0.060374,0.011021
8,SY,1119,39.993111,53.985574,26.598272,56.940070,39.744617,53.906955,26.463196,56.874011,-0.248494,-0.078619
9,TN,1116,29.285225,43.452865,17.062352,46.059624,29.246732,43.517582,16.922631,46.152194,-0.038493,0.064717


Text-level overrides: 761/12,250
OOF spBLEU gain: -0.073854 | deployment: 92_mixed_best_checkpoint_variant_per_country

OFFICIAL-STYLE DEVELOPMENT RESULT
System: 92_mixed_best_checkpoint_variant_per_country
Average spBLEU: 30.928003
Average chrF++: 45.594526

SUBMIT THIS ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/95_qwen3_06b_selective_uplift_cleanlabels_v1/submission_predictions.zip


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,32.903189,46.892198,19.510451,49.894903
1,JO,1113,35.301943,49.377895,19.819243,52.901916
2,LB,1118,31.701457,45.871215,20.021014,48.586333
3,MA,1110,23.530143,39.376601,12.865149,42.236961
4,MR,1114,17.380438,33.733013,7.100699,37.779153
5,OM,1109,36.147408,49.843410,20.805106,53.247687
6,PS,1110,33.418402,47.762506,20.699325,51.031341
7,SA,1110,33.166094,48.135685,18.903768,52.048449
8,SY,1119,39.993111,53.985574,26.598272,56.940070
9,TN,1116,29.285225,43.452865,17.062352,46.059624


### **Post Editing System**

In [7]:
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.51,<5",
    "peft>=0.17,<1",
    "accelerate>=1,<2",
    "safetensors",
    "sentencepiece",
    "scikit-learn",
    "sacrebleu>=2.4,<3"
])

print("Dependencies are ready. Restart the kernel.")

Dependencies are ready. Restart the kernel.


In [8]:
import os, gc, re, ast, json, math, random, hashlib, zipfile
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import sacrebleu
import torch
import torch.nn as nn

from IPython.display import display
from peft import (
    LoraConfig,
    PeftModel,
    TaskType,
    get_peft_model,
    get_peft_model_state_dict,
    set_peft_model_state_dict
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedGroupKFold
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_cosine_schedule_with_warmup
)

RUN_NAME = "96_nilechat3b_multicandidate_posteditor_deadline_v1"
PE_RUN_DIR = PROJECT_DIR / "posteditors" / RUN_NAME
PE_OUTPUT_DIR = INFERENCE_VARIANTS_ROOT / RUN_NAME

PE_BASE_MODEL_DIR = (
    PROJECT_DIR
    / "models"
    / "hf"
    / "NileChat-3B-Base"
)

PE_TRAINING_RUN_DIR = (
    PROJECT_DIR
    / "runs"
    / "nilechat3b_all14"
    / (
        "nilechat3b_alexandria_all14_"
        "context3_complete2shot_all_group_r16_alpha32_"
        "3epochs_beam4_nonquant_server5090_v1"
    )
)

PE_TRANSLATOR_CHECKPOINT = (
    PE_TRAINING_RUN_DIR
    / "checkpoint-16600"
)

# ============================================================
# ALL TRAINING PARAMETERS ARE HERE
# Deadline configuration: two folds and one epoch.
# ============================================================

POSTEDIT_N_FOLDS = 2
POSTEDIT_EPOCHS = 1

POSTEDIT_MAX_LENGTH = 768
POSTEDIT_MAX_TARGET_TOKENS = 160
POSTEDIT_MAX_NEW_TOKENS = 128

POSTEDIT_BATCH_SIZE = 1
POSTEDIT_GRAD_ACCUM_STEPS = 16
POSTEDIT_LR = 1e-4
POSTEDIT_WARMUP_RATIO = 0.03
POSTEDIT_WEIGHT_DECAY = 0.01
POSTEDIT_MAX_GRAD_NORM = 1.0
POSTEDIT_SAVE_STEPS = 100

POSTEDIT_GENERATION_BATCH_SIZE = 4
POSTEDIT_SAVE_ROWS = 64

POSTEDIT_LORA_R = 16
POSTEDIT_LORA_ALPHA = 32
POSTEDIT_LORA_DROPOUT = 0.05

GATE_CALIBRATION_FRACTION = 0.20
GATE_LABEL_MIN_UTILITY_GAIN = 0.25
GATE_MIN_ACCEPTS = 10
GATE_THRESHOLD_POINTS = 41

DEPLOY_MIN_SPBLEU_GAIN = 0.0

PE_SEED = SEED
PE_DEVICE = torch.device("cuda")
PE_USE_BF16 = torch.cuda.is_bf16_supported()
PE_DTYPE = (
    torch.bfloat16
    if PE_USE_BF16
    else torch.float16
)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required.")

for required_path in [
    PE_BASE_MODEL_DIR,
    PE_TRANSLATOR_CHECKPOINT
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

if not (
    PE_TRANSLATOR_CHECKPOINT
    / "adapter_config.json"
).exists():
    raise FileNotFoundError(
        f"Translator adapter missing: "
        f"{PE_TRANSLATOR_CHECKPOINT}"
    )

PE_RUN_DIR.mkdir(parents=True, exist_ok=True)
PE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def pe_hash(value):
    payload = json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False
    ).encode()

    return hashlib.sha256(
        payload
    ).hexdigest()[:20]

PE_SIGNATURE = pe_hash({
    "run": RUN_NAME,
    "candidate_hash": CANDIDATE_HASH,
    "base": str(PE_BASE_MODEL_DIR),
    "translator": str(PE_TRANSLATOR_CHECKPOINT),
    "folds": POSTEDIT_N_FOLDS,
    "epochs": POSTEDIT_EPOCHS,
    "max_length": POSTEDIT_MAX_LENGTH,
    "target_tokens": POSTEDIT_MAX_TARGET_TOKENS,
    "batch": POSTEDIT_BATCH_SIZE,
    "accum": POSTEDIT_GRAD_ACCUM_STEPS,
    "lr": POSTEDIT_LR,
    "lora": [
        POSTEDIT_LORA_R,
        POSTEDIT_LORA_ALPHA,
        POSTEDIT_LORA_DROPOUT
    ],
    "seed": PE_SEED
})

print("Run:", RUN_NAME)
print(
    "GPU:",
    torch.cuda.get_device_name(0),
    "| dtype:",
    PE_DTYPE
)
print("Translator:", PE_TRANSLATOR_CHECKPOINT)
print(
    "TRAINING PARAMETERS: folds=2, epochs=1, "
    "batch=1, grad_accum=16, LR=1e-4, "
    "max_length=768, LoRA r=16"
)
print("Signature:", PE_SIGNATURE)

Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


Run: 96_nilechat3b_multicandidate_posteditor_deadline_v1
GPU: NVIDIA GeForce RTX 5090 | dtype: torch.bfloat16
Translator: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_all14/nilechat3b_alexandria_all14_context3_complete2shot_all_group_r16_alpha32_3epochs_beam4_nonquant_server5090_v1/checkpoint-16600
TRAINING PARAMETERS: folds=2, epochs=1, batch=1, grad_accum=16, LR=1e-4, max_length=768, LoRA r=16
Signature: a2fd89987ae9cdb62768


In [9]:
def pe_atomic_torch_save(obj, path):
    path = Path(path)
    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )

    torch.save(obj, temporary)
    os.replace(temporary, path)

def pe_atomic_json_save(obj, path):
    path = Path(path)
    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )

    temporary.write_text(
        json.dumps(
            obj,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    os.replace(temporary, path)

def pe_atomic_csv_save(frame, path):
    path = Path(path)
    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )

    frame.to_csv(
        temporary,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(temporary, path)

def pe_load_torch(path):
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )
    except TypeError:
        return torch.load(
            path,
            map_location="cpu"
        )

def pe_scaler():
    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=not PE_USE_BF16
        )
    except TypeError:
        return torch.cuda.amp.GradScaler(
            enabled=not PE_USE_BF16
        )

def pe_optimizer_to(optimizer, device):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)

def pe_capture_rng():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all()
    }

def pe_restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    torch.cuda.set_rng_state_all(
        state["cuda"]
    )

def pe_norm(value):
    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()

pe_groups = (
    official_dev_df["config"].astype(str)
    + "::"
    + official_dev_df[
        "conversation_id"
    ].astype(str)
).to_numpy()

post_fold_id = np.full(
    N,
    -1,
    dtype=np.int8
)

pe_splitter = StratifiedGroupKFold(
    n_splits=POSTEDIT_N_FOLDS,
    shuffle=True,
    random_state=PE_SEED
)

for fold, (_, heldout_indices) in enumerate(
    pe_splitter.split(
        np.zeros(N),
        official_dev_df["config"],
        pe_groups
    )
):
    post_fold_id[
        heldout_indices
    ] = fold

if (post_fold_id < 0).any():
    raise RuntimeError(
        "Post-editor fold assignment is incomplete."
    )

fold_leakage = (
    official_dev_df
    .assign(pe_fold=post_fold_id)
    .groupby([
        "config",
        "conversation_id"
    ])["pe_fold"]
    .nunique()
    .max()
)

if fold_leakage != 1:
    raise RuntimeError(
        "Conversation leakage in post-editor folds."
    )

display(pd.crosstab(
    official_dev_df["config"],
    post_fold_id,
    margins=True
))

col_0,0,1,All
config,,,
EG,556,557,1113
JO,557,556,1113
LB,560,558,1118
MA,555,555,1110
MR,557,557,1114
OM,555,554,1109
PS,555,555,1110
SA,556,554,1110
SY,559,560,1119


In [10]:
def pe_corpus_spbleu(
    predictions,
    references
):
    return sacrebleu.corpus_bleu(
        list(map(str, predictions)),
        [list(map(str, references))],
        tokenize="flores200"
    ).score

def pe_choose_alternatives(
    train_indices
):
    mapping = {}
    audit_rows = []

    configs = (
        official_dev_df["config"]
        .to_numpy()
    )

    references = (
        official_dev_df[
            "reference_arabic"
        ]
        .astype(str)
        .to_numpy()
    )

    for country in sorted(
        set(configs)
    ):
        rows = np.asarray([
            index
            for index in train_indices
            if configs[index] == country
        ], dtype=np.int64)

        ranked = []

        for candidate_index, variant_name in enumerate(
            variant_names
        ):
            hypotheses = candidate_texts[
                rows,
                candidate_index
            ]

            changed_rate = np.mean([
                pe_norm(hypothesis)
                != pe_norm(baseline)
                for hypothesis, baseline
                in zip(
                    hypotheses,
                    baseline_predictions[rows]
                )
            ])

            score = pe_corpus_spbleu(
                hypotheses,
                references[rows]
            )

            ranked.append((
                candidate_index,
                variant_name,
                score,
                changed_rate
            ))

        ranked.sort(
            key=lambda item: (
                item[2],
                item[3]
            ),
            reverse=True
        )

        eligible = [
            item
            for item in ranked
            if item[3] >= 0.01
        ]

        selected = (
            eligible
            + [
                item
                for item in ranked
                if item not in eligible
            ]
        )[:2]

        mapping[country] = [
            int(item[0])
            for item in selected
        ]

        for alternative_rank, item in enumerate(
            selected,
            1
        ):
            audit_rows.append({
                "country": country,
                "alternative_rank": (
                    alternative_rank
                ),
                "variant_index": item[0],
                "variant": item[1],
                "train_spBLEU": item[2],
                "changed_from_system92_rate": (
                    item[3]
                )
            })

    return (
        mapping,
        pd.DataFrame(audit_rows)
    )

fold_alternative_maps = {}

for fold in range(
    POSTEDIT_N_FOLDS
):
    train_indices = np.where(
        post_fold_id != fold
    )[0]

    mapping, audit = (
        pe_choose_alternatives(
            train_indices
        )
    )

    fold_alternative_maps[
        fold
    ] = mapping

    fold_dir = (
        PE_RUN_DIR
        / f"fold_{fold}"
    )

    fold_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    audit.to_csv(
        fold_dir
        / "alternative_selection.csv",
        index=False,
        encoding="utf-8-sig"
    )

    pe_atomic_json_save({
        "signature": PE_SIGNATURE,
        "mapping": mapping
    }, (
        fold_dir
        / "alternative_selection.json"
    ))

    print(
        f"Fold {fold + 1} alternatives"
    )

    display(audit)

Fold 1 alternatives


,country,alternative_rank,variant_index,variant,train_spBLEU,changed_from_system92_rate
0,EG,1,4,04_training_parity_with_participants,31.882971,0.533214
1,EG,2,1,01_exact_training_parity,31.773442,0.572711
2,JO,1,7,07_ckpt16000_retrieved_two_shot,35.521669,0.760791
3,JO,2,2,02_metadata_no_shots,35.382300,0.514388
4,LB,1,3,03_retrieved_two_shot,31.791283,0.815412
5,LB,2,2,02_metadata_no_shots,31.729423,0.840502
6,MA,1,3,03_retrieved_two_shot,23.486048,0.520721
7,MA,2,4,04_training_parity_with_participants,23.192581,0.677477
8,MR,1,5,05_retrieved_two_shot_with_participants,17.016047,0.933573
9,MR,2,2,02_metadata_no_shots,16.907964,0.924596


Fold 2 alternatives


,country,alternative_rank,variant_index,variant,train_spBLEU,changed_from_system92_rate
0,EG,1,1,01_exact_training_parity,33.955475,0.507194
1,EG,2,3,03_retrieved_two_shot,33.794566,0.440647
2,JO,1,5,05_retrieved_two_shot_with_participants,34.978901,0.450628
3,JO,2,2,02_metadata_no_shots,34.893444,0.569120
4,LB,1,5,05_retrieved_two_shot_with_participants,31.298422,0.819643
5,LB,2,3,03_retrieved_two_shot,31.150013,0.812500
6,MA,1,2,02_metadata_no_shots,23.853319,0.747748
7,MA,2,3,03_retrieved_two_shot,23.521616,0.542342
8,MR,1,3,03_retrieved_two_shot,17.964300,0.922801
9,MR,2,0,00_previous_official_control,17.570105,0.924596


In [11]:
PE_SYSTEM_PROMPT = (
    "You are a conservative automatic post-editor "
    "for English-to-dialectal-Arabic dialogue "
    "translation. Use the English source and context "
    "as the authority. Start from System92, borrow "
    "only useful wording from the alternatives, "
    "preserve every fact, and write naturally in the "
    "requested country dialect. Return only the final "
    "Arabic translation."
)

def pe_context(value):
    if isinstance(value, str):
        try:
            value = ast.literal_eval(
                value
            )
        except Exception:
            value = []

    if not isinstance(value, list):
        return "None"

    parts = []

    for turn in value[-3:]:
        if not isinstance(turn, dict):
            continue

        speaker = pe_norm(
            turn.get("speaker", "")
        )

        direction = pe_norm(
            turn.get("direction", "")
        )

        text = pe_norm(
            turn.get("text", "")
        )

        prefix = "/".join(
            item
            for item in [
                speaker,
                direction
            ]
            if item
        )

        if text:
            parts.append(
                f"{prefix}: {text}"
                if prefix
                else text
            )

    return (
        " <turn> ".join(parts)
        if parts
        else "None"
    )

def pe_alternative_indices(
    row_index,
    fold
):
    country = str(
        official_dev_df.iloc[
            int(row_index)
        ]["config"]
    )

    return fold_alternative_maps[
        int(fold)
    ][country]

def pe_prompt(
    row_index,
    fold
):
    row_index = int(row_index)

    row = official_dev_df.iloc[
        row_index
    ]

    alt1, alt2 = (
        pe_alternative_indices(
            row_index,
            fold
        )
    )

    code = str(row["config"])

    dialect = DIALECT_NAMES.get(
        code,
        code
    )

    baseline_variant = variant_names[
        int(baseline_idx[row_index])
    ]

    return (
        f"### System:\n"
        f"{PE_SYSTEM_PROMPT}\n\n"

        f"### Instruction:\n"
        f"Requested dialect: "
        f"{code} ({dialect})\n"

        f"Domain: "
        f"{pe_norm(row.get('domain', ''))}\n"

        f"Speaker: "
        f"{pe_norm(row.get('speaker', ''))}\n"

        f"Gender direction: "
        f"{pe_norm(row.get('gender_direction', ''))}\n"

        f"Previous English turns: "
        f"{pe_context(row.get('previous_english_turns', []))}\n"

        f"Current English source: "
        f"{pe_norm(row['source_text'])}\n"

        f"System92 [{baseline_variant}]: "
        f"{pe_norm(baseline_predictions[row_index])}\n"

        f"Alternative 1 [{variant_names[alt1]}]: "
        f"{pe_norm(candidate_texts[row_index, alt1])}\n"

        f"Alternative 2 [{variant_names[alt2]}]: "
        f"{pe_norm(candidate_texts[row_index, alt2])}\n\n"

        f"### Arabic translation:\n"
    )

pe_tokenizer = AutoTokenizer.from_pretrained(
    str(PE_BASE_MODEL_DIR),
    trust_remote_code=True,
    local_files_only=True,
    use_fast=True,
    extra_special_tokens={}
)

if pe_tokenizer.pad_token_id is None:
    pe_tokenizer.pad_token = pe_tokenizer.eos_token

pe_tokenizer.save_pretrained(
    PE_RUN_DIR / "tokenizer"
)

print("Tokenizer loaded:", type(pe_tokenizer).__name__)
print("Vocabulary size:", len(pe_tokenizer))
print("EOS token:", pe_tokenizer.eos_token)
print("PAD token:", pe_tokenizer.pad_token)


class PostEditDataset(Dataset):
    def __init__(
        self,
        indices,
        fold
    ):
        self.indices = np.asarray(
            indices,
            dtype=np.int64
        )

        self.fold = int(fold)

    def __len__(self):
        return len(self.indices)

    def __getitem__(
        self,
        position
    ):
        row_index = int(
            self.indices[position]
        )

        target = pe_norm(
            official_dev_df.iloc[
                row_index
            ]["reference_arabic"]
        )

        return (
            row_index,
            pe_prompt(
                row_index,
                self.fold
            ),
            target
        )

class PostEditCollator:
    def __call__(
        self,
        items
    ):
        pe_tokenizer.padding_side = "right"
        pe_tokenizer.truncation_side = "right"

        rows = []
        sequences = []
        labels = []

        for (
            row_index,
            prompt,
            target
        ) in items:
            prompt_ids = pe_tokenizer(
                prompt,
                add_special_tokens=True,
                truncation=False
            )["input_ids"]

            target_ids = pe_tokenizer(
                target,
                add_special_tokens=False,
                truncation=True,
                max_length=(
                    POSTEDIT_MAX_TARGET_TOKENS
                )
            )["input_ids"]

            target_ids = (
                target_ids
                + [
                    pe_tokenizer.eos_token_id
                ]
            )

            available_prompt_length = max(
                1,
                POSTEDIT_MAX_LENGTH
                - len(target_ids)
            )

            prompt_ids = prompt_ids[
                -available_prompt_length:
            ]

            input_ids = (
                prompt_ids
                + target_ids
            )

            rows.append(row_index)
            sequences.append(input_ids)

            labels.append(
                [-100] * len(prompt_ids)
                + target_ids
            )

        max_length = max(
            map(len, sequences)
        )

        pad_token_id = (
            pe_tokenizer.pad_token_id
        )

        input_ids = []
        attention_mask = []
        padded_labels = []

        for ids, target_labels in zip(
            sequences,
            labels
        ):
            padding_width = (
                max_length
                - len(ids)
            )

            input_ids.append(
                ids
                + [pad_token_id]
                * padding_width
            )

            attention_mask.append(
                [1] * len(ids)
                + [0] * padding_width
            )

            padded_labels.append(
                target_labels
                + [-100]
                * padding_width
            )

        return {
            "row_indices": torch.tensor(
                rows,
                dtype=torch.long
            ),
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                attention_mask,
                dtype=torch.long
            ),
            "labels": torch.tensor(
                padded_labels,
                dtype=torch.long
            )
        }

sample_fold = int(
    post_fold_id[0]
)

print(
    pe_prompt(
        0,
        sample_fold
    )
)

print(
    "TARGET:",
    official_dev_df.iloc[
        0
    ]["reference_arabic"]
)

Tokenizer loaded: Qwen2TokenizerFast
Vocabulary size: 151665
EOS token: <|endoftext|>
PAD token: <|endoftext|>
### System:
You are a conservative automatic post-editor for English-to-dialectal-Arabic dialogue translation. Use the English source and context as the authority. Start from System92, borrow only useful wording from the alternatives, preserve every fact, and write naturally in the requested country dialect. Return only the final Arabic translation.

### Instruction:
Requested dialect: EG (Egyptian Arabic)
Domain: Agriculture and farming
Speaker: Waterresourcemanager
Gender direction: female -> female
Previous English turns: None
Current English source: Good news, the ministry has finally approved the budget for the barrage repairs.
System92 [05_retrieved_two_shot_with_participants]: أخبار حلوة، الوزارة أخيرا وافقت على ميزانية إصلاح السد.
Alternative 1 [01_exact_training_parity]: في أخبار حلوة، الوزارة أخيرا وافقت على ميزانية إصلاح السد.
Alternative 2 [03_retrieved_two_shot]: 

In [12]:
PE_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
]

def pe_load_merged_translator():
    gc.collect()
    torch.cuda.empty_cache()

    base_model = (
        AutoModelForCausalLM
        .from_pretrained(
            str(PE_BASE_MODEL_DIR),
            dtype=PE_DTYPE,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
            local_files_only=True,
            attn_implementation="sdpa"
        )
    )

    translator = (
        PeftModel
        .from_pretrained(
            base_model,
            str(
                PE_TRANSLATOR_CHECKPOINT
            ),
            is_trainable=False
        )
    )

    model = (
        translator
        .merge_and_unload(
            safe_merge=True
        )
    )

    del translator
    del base_model

    return model

def pe_create_model(
    final_adapter=None
):
    model = (
        pe_load_merged_translator()
    )

    if final_adapter is not None:
        model = (
            PeftModel
            .from_pretrained(
                model,
                str(final_adapter),
                is_trainable=False
            )
        )

        model.config.use_cache = True

    else:
        model.config.use_cache = False

        if hasattr(
            model,
            "gradient_checkpointing_enable"
        ):
            model.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={
                    "use_reentrant": False
                }
            )

        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=POSTEDIT_LORA_R,
            lora_alpha=(
                POSTEDIT_LORA_ALPHA
            ),
            lora_dropout=(
                POSTEDIT_LORA_DROPOUT
            ),
            target_modules=(
                PE_TARGET_MODULES
            ),
            bias="none"
        )

        model = get_peft_model(
            model,
            lora_config
        )

        if hasattr(
            model,
            "enable_input_require_grads"
        ):
            model.enable_input_require_grads()

    return model.to(
        PE_DEVICE
    )

print(
    "Model builder ready. "
    "Checkpoint 16600 is merged before "
    "the post-editing LoRA is attached."
)

Model builder ready. Checkpoint 16600 is merged before the post-editing LoRA is attached.


In [13]:
def pe_adapter_state(model):
    return {
        key: value.detach().cpu()
        for key, value
        in get_peft_model_state_dict(
            model
        ).items()
    }

def pe_save_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    next_batch,
    global_step
):
    pe_atomic_torch_save({
        "signature": PE_SIGNATURE,
        "adapter": pe_adapter_state(
            model
        ),
        "optimizer": (
            optimizer.state_dict()
        ),
        "scheduler": (
            scheduler.state_dict()
        ),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "next_batch": next_batch,
        "global_step": global_step,
        "rng": pe_capture_rng()
    }, path)

def pe_train_fold(fold):
    fold_dir = (
        PE_RUN_DIR
        / f"fold_{fold}"
    )

    final_dir = (
        fold_dir
        / "adapter_final"
    )

    last_path = (
        fold_dir
        / "checkpoint_last.pt"
    )

    training_metadata_path = (
        fold_dir
        / "training_complete.json"
    )

    train_indices = np.where(
        post_fold_id != fold
    )[0]

    if (
        final_dir
        / "adapter_config.json"
    ).exists():
        metadata = json.loads(
            training_metadata_path
            .read_text(
                encoding="utf-8"
            )
        )

        if (
            metadata["signature"]
            != PE_SIGNATURE
        ):
            raise RuntimeError(
                f"Fold {fold} final adapter "
                "signature mismatch. "
                "Change RUN_NAME."
            )

        print(
            f"Fold {fold + 1}: "
            "loading completed adapter"
        )

        return pe_create_model(
            final_dir
        )

    model = pe_create_model()

    model.print_trainable_parameters()

    dataset = PostEditDataset(
        train_indices,
        fold
    )

    batches_per_epoch = math.ceil(
        len(dataset)
        / POSTEDIT_BATCH_SIZE
    )

    updates_per_epoch = math.ceil(
        batches_per_epoch
        / POSTEDIT_GRAD_ACCUM_STEPS
    )

    total_updates = (
        updates_per_epoch
        * POSTEDIT_EPOCHS
    )

    optimizer = torch.optim.AdamW(
        [
            parameter
            for parameter
            in model.parameters()
            if parameter.requires_grad
        ],
        lr=POSTEDIT_LR,
        weight_decay=(
            POSTEDIT_WEIGHT_DECAY
        )
    )

    scheduler = (
        get_cosine_schedule_with_warmup(
            optimizer,
            int(
                total_updates
                * POSTEDIT_WARMUP_RATIO
            ),
            total_updates
        )
    )

    scaler = pe_scaler()

    start_epoch = 0
    start_batch = 0
    global_step = 0

    if last_path.exists():
        state = pe_load_torch(
            last_path
        )

        if (
            state["signature"]
            != PE_SIGNATURE
        ):
            raise RuntimeError(
                f"Fold {fold} resume "
                "signature mismatch. "
                "Change RUN_NAME."
            )

        set_peft_model_state_dict(
            model,
            state["adapter"]
        )

        optimizer.load_state_dict(
            state["optimizer"]
        )

        pe_optimizer_to(
            optimizer,
            PE_DEVICE
        )

        scheduler.load_state_dict(
            state["scheduler"]
        )

        scaler.load_state_dict(
            state["scaler"]
        )

        start_epoch = state["epoch"]
        start_batch = state["next_batch"]
        global_step = state["global_step"]

        pe_restore_rng(
            state["rng"]
        )

        print(
            f"Fold {fold + 1}: resumed "
            f"epoch={start_epoch}, "
            f"batch={start_batch}, "
            f"step={global_step}"
        )

    for epoch in range(
        start_epoch,
        POSTEDIT_EPOCHS
    ):
        generator = (
            torch.Generator()
            .manual_seed(
                PE_SEED
                + 1000 * fold
                + epoch
            )
        )

        loader = DataLoader(
            dataset,
            batch_size=(
                POSTEDIT_BATCH_SIZE
            ),
            shuffle=True,
            generator=generator,
            collate_fn=PostEditCollator(),
            num_workers=0,
            pin_memory=True
        )

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        running_loss = 0.0

        progress = tqdm(
            enumerate(loader),
            total=len(loader),
            desc=(
                f"Post-edit fold "
                f"{fold + 1}/"
                f"{POSTEDIT_N_FOLDS} "
                f"epoch {epoch + 1}/"
                f"{POSTEDIT_EPOCHS}"
            )
        )

        for (
            batch_index,
            batch
        ) in progress:
            if (
                epoch == start_epoch
                and batch_index
                < start_batch
            ):
                continue

            batch.pop(
                "row_indices"
            )

            batch = {
                key: value.to(
                    PE_DEVICE,
                    non_blocking=True
                )
                for key, value
                in batch.items()
            }

            with torch.autocast(
                "cuda",
                dtype=PE_DTYPE
            ):
                loss = model(
                    **batch,
                    use_cache=False
                ).loss

            scaled_loss = (
                loss
                / POSTEDIT_GRAD_ACCUM_STEPS
            )

            scaler.scale(
                scaled_loss
            ).backward()

            running_loss += float(
                loss.detach()
            )

            do_step = (
                (
                    batch_index + 1
                )
                % POSTEDIT_GRAD_ACCUM_STEPS
                == 0
                or batch_index + 1
                == len(loader)
            )

            if do_step:
                scaler.unscale_(
                    optimizer
                )

                nn.utils.clip_grad_norm_(
                    model.parameters(),
                    POSTEDIT_MAX_GRAD_NORM
                )

                scaler.step(
                    optimizer
                )

                scaler.update()
                scheduler.step()

                optimizer.zero_grad(
                    set_to_none=True
                )

                global_step += 1

                progress.set_postfix(
                    loss=(
                        f"{running_loss / max(1, batch_index + 1):.4f}"
                    ),
                    step=global_step
                )

                if (
                    global_step
                    % POSTEDIT_SAVE_STEPS
                    == 0
                ):
                    pe_save_checkpoint(
                        last_path,
                        model,
                        optimizer,
                        scheduler,
                        scaler,
                        epoch,
                        batch_index + 1,
                        global_step
                    )

        start_batch = 0

        pe_save_checkpoint(
            last_path,
            model,
            optimizer,
            scheduler,
            scaler,
            epoch + 1,
            0,
            global_step
        )

    final_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    model.save_pretrained(
        final_dir,
        safe_serialization=True
    )

    pe_atomic_json_save({
        "signature": PE_SIGNATURE,
        "fold": fold,
        "train_rows": len(
            train_indices
        ),
        "global_step": global_step
    }, training_metadata_path)

    print(
        f"Fold {fold + 1}: "
        f"saved {final_dir}"
    )

    return model

In [14]:
def pe_clean_generation(text):
    text = (
        str(text)
        .replace(
            "<|endoftext|>",
            ""
        )
        .replace(
            "<|im_end|>",
            ""
        )
        .strip()
    )

    text = re.split(
        r"\n\s*###",
        text,
        maxsplit=1
    )[0].strip()

    text = re.sub(
        r"^(Arabic translation|الترجمة العربية)\s*:\s*",
        "",
        text,
        flags=re.I
    ).strip()

    return text

def pe_generate_batch(
    model,
    row_indices,
    fold
):
    prompts = [
        pe_prompt(
            int(row_index),
            fold
        )
        for row_index
        in row_indices
    ]

    pe_tokenizer.padding_side = "left"
    pe_tokenizer.truncation_side = "left"

    encoded = pe_tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=(
            POSTEDIT_MAX_LENGTH
            - POSTEDIT_MAX_NEW_TOKENS
        ),
        return_tensors="pt",
        add_special_tokens=True
    ).to(PE_DEVICE)

    model.eval()
    model.config.use_cache = True

    with torch.inference_mode():
        output = model.generate(
            **encoded,
            max_new_tokens=(
                POSTEDIT_MAX_NEW_TOKENS
            ),
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.05,
            eos_token_id=(
                pe_tokenizer.eos_token_id
            ),
            pad_token_id=(
                pe_tokenizer.pad_token_id
            ),
            use_cache=True,
            return_dict_in_generate=True,
            output_scores=True
        )

    generated = output.sequences[
        :,
        encoded["input_ids"].shape[1]:
    ]

    logprob_sums = torch.zeros(
        len(row_indices),
        device=PE_DEVICE
    )

    token_counts = torch.zeros(
        len(row_indices),
        device=PE_DEVICE
    )

    finished = torch.zeros(
        len(row_indices),
        dtype=torch.bool,
        device=PE_DEVICE
    )

    for step, logits in enumerate(
        output.scores
    ):
        token = generated[
            :,
            step
        ]

        active = ~finished

        token_logprob = (
            torch.log_softmax(
                logits.float(),
                dim=-1
            )
            .gather(
                1,
                token[:, None]
            )
            .squeeze(1)
        )

        logprob_sums += (
            token_logprob
            * active
        )

        token_counts += active

        finished |= token.eq(
            pe_tokenizer.eos_token_id
        )

    confidence = (
        logprob_sums
        / token_counts.clamp_min(1)
    ).cpu().numpy()

    predictions = [
        pe_clean_generation(
            pe_tokenizer.decode(
                tokens,
                skip_special_tokens=True
            )
        )
        for tokens in generated
    ]

    for position, row_index in enumerate(
        row_indices
    ):
        if not predictions[position]:
            predictions[position] = str(
                baseline_predictions[
                    int(row_index)
                ]
            )

            confidence[position] = -99.0

    return (
        predictions,
        confidence
    )

def pe_score_fold(
    model,
    fold
):
    fold_dir = (
        PE_RUN_DIR
        / f"fold_{fold}"
    )

    cache_path = (
        fold_dir
        / "heldout_postedits.csv"
    )

    heldout_indices = np.where(
        post_fold_id == fold
    )[0]

    expected_ids = (
        official_dev_df.iloc[
            heldout_indices
        ]["source_id"]
        .astype(str)
        .tolist()
    )

    if cache_path.exists():
        cache = pd.read_csv(
            cache_path
        )

        cache["source_id"] = (
            cache["source_id"]
            .astype(str)
        )

        unexpected_ids = (
            set(cache["source_id"])
            - set(expected_ids)
        )

        if unexpected_ids:
            raise RuntimeError(
                f"Fold {fold} generation "
                "cache mismatch."
            )

    else:
        cache = pd.DataFrame(
            columns=[
                "source_id",
                "row_index",
                "prediction",
                "mean_logprob",
                "alt1_idx",
                "alt2_idx"
            ]
        )

    complete_ids = set(
        cache["source_id"]
        .astype(str)
    )

    pending_indices = [
        index
        for index in heldout_indices
        if str(
            official_dev_df.iloc[
                index
            ]["source_id"]
        ) not in complete_ids
    ]

    for start in tqdm(
        range(
            0,
            len(pending_indices),
            POSTEDIT_GENERATION_BATCH_SIZE
        ),
        desc=(
            f"Post-edit fold "
            f"{fold + 1} "
            "held-out generation"
        )
    ):
        row_indices = pending_indices[
            start:
            start
            + POSTEDIT_GENERATION_BATCH_SIZE
        ]

        try:
            predictions, logprobs = (
                pe_generate_batch(
                    model,
                    row_indices,
                    fold
                )
            )

        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()

            predictions = []
            logprobs = []

            for row_index in row_indices:
                one_prediction, one_logprob = (
                    pe_generate_batch(
                        model,
                        [row_index],
                        fold
                    )
                )

                predictions.extend(
                    one_prediction
                )

                logprobs.extend(
                    one_logprob
                )

        records = []

        for (
            row_index,
            prediction,
            logprob
        ) in zip(
            row_indices,
            predictions,
            logprobs
        ):
            alt1, alt2 = (
                pe_alternative_indices(
                    row_index,
                    fold
                )
            )

            records.append({
                "source_id": str(
                    official_dev_df.iloc[
                        row_index
                    ]["source_id"]
                ),
                "row_index": row_index,
                "prediction": prediction,
                "mean_logprob": float(
                    logprob
                ),
                "alt1_idx": alt1,
                "alt2_idx": alt2
            })

        cache = pd.concat([
            cache,
            pd.DataFrame(records)
        ], ignore_index=True)

        if (
            len(cache)
            % POSTEDIT_SAVE_ROWS
            < len(records)
        ):
            pe_atomic_csv_save(
                cache,
                cache_path
            )

    pe_atomic_csv_save(
        cache,
        cache_path
    )

    ordered = pd.DataFrame({
        "source_id": expected_ids,
        "row_index": heldout_indices
    }).merge(
        cache,
        on=[
            "source_id",
            "row_index"
        ],
        how="left",
        validate="one_to_one"
    )

    if ordered[
        "prediction"
    ].isna().any():
        raise RuntimeError(
            f"Fold {fold} generation "
            "is incomplete."
        )

    return ordered

In [15]:
oof_postedits = np.full(
    N,
    "",
    dtype=object
)

oof_logprobs = np.full(
    N,
    np.nan,
    dtype=np.float32
)

oof_alt1_idx = np.full(
    N,
    -1,
    dtype=np.int16
)

oof_alt2_idx = np.full(
    N,
    -1,
    dtype=np.int16
)

for fold in range(
    POSTEDIT_N_FOLDS
):
    print(
        "\n"
        + "=" * 90
    )

    print(
        f"POST-EDITOR OUTER FOLD "
        f"{fold + 1}/"
        f"{POSTEDIT_N_FOLDS}"
    )

    print("=" * 90)

    pe_model = pe_train_fold(
        fold
    )

    fold_output = pe_score_fold(
        pe_model,
        fold
    )

    row_indices = (
        fold_output["row_index"]
        .to_numpy(np.int64)
    )

    oof_postedits[
        row_indices
    ] = (
        fold_output["prediction"]
        .astype(str)
        .to_numpy()
    )

    oof_logprobs[
        row_indices
    ] = (
        fold_output["mean_logprob"]
        .to_numpy(np.float32)
    )

    oof_alt1_idx[
        row_indices
    ] = (
        fold_output["alt1_idx"]
        .to_numpy(np.int16)
    )

    oof_alt2_idx[
        row_indices
    ] = (
        fold_output["alt2_idx"]
        .to_numpy(np.int16)
    )

    del pe_model

    gc.collect()
    torch.cuda.empty_cache()

if (
    oof_postedits == ""
).any():
    raise RuntimeError(
        "OOF post-edit predictions "
        "are incomplete."
    )

if not np.isfinite(
    oof_logprobs
).all():
    raise RuntimeError(
        "OOF confidence values "
        "are incomplete."
    )

np.save(
    PE_OUTPUT_DIR
    / "oof_postedit_predictions.npy",
    oof_postedits
)

np.save(
    PE_OUTPUT_DIR
    / "oof_postedit_logprobs.npy",
    oof_logprobs
)

print(
    "Complete OOF post-edits:",
    len(oof_postedits)
)


POST-EDITOR OUTER FOLD 1/2
Fold 1: loading completed adapter


Post-edit fold 1 held-out generation: 0it [00:00, ?it/s]


POST-EDITOR OUTER FOLD 2/2
Fold 2: loading completed adapter


Post-edit fold 2 held-out generation: 0it [00:00, ?it/s]

Complete OOF post-edits: 12250


In [16]:
import gc
import hashlib
import math
import os

import numpy as np

from difflib import SequenceMatcher
from sacrebleu.metrics import BLEU, CHRF
from tqdm.auto import tqdm


# =============================================================================
# Validate PE-8 outputs
# =============================================================================

if not (
    len(baseline_predictions)
    == len(oof_postedits)
    == len(oof_logprobs)
    == len(oof_alt1_idx)
    == len(oof_alt2_idx)
    == N
):
    raise RuntimeError(
        "PE-8 output lengths do not match N."
    )

if (
    (oof_postedits == "").any()
    or not np.isfinite(oof_logprobs).all()
):
    raise RuntimeError(
        "PE-8 predictions or confidence values are incomplete."
    )

number_of_candidates = candidate_texts.shape[1]

invalid_alternative_indices = (
    (oof_alt1_idx < 0)
    | (oof_alt2_idx < 0)
    | (oof_alt1_idx >= number_of_candidates)
    | (oof_alt2_idx >= number_of_candidates)
)

if invalid_alternative_indices.any():
    bad_rows = np.where(
        invalid_alternative_indices
    )[0][:10]

    raise RuntimeError(
        "Invalid PE-8 alternative indices at rows: "
        f"{bad_rows.tolist()}"
    )


# =============================================================================
# Create each SacreBLEU metric exactly once
# =============================================================================

pe_sentence_bleu_metric = BLEU(
    tokenize="flores200",
    smooth_method="exp",
    effective_order=True
)

pe_sentence_chrf_metric = CHRF(
    word_order=2
)


def pe_sentence_utility(
    hypothesis,
    reference
):
    hypothesis = str(hypothesis)
    reference = str(reference)

    bleu = pe_sentence_bleu_metric.sentence_score(
        hypothesis,
        [reference]
    ).score

    chrf = pe_sentence_chrf_metric.sentence_score(
        hypothesis,
        [reference]
    ).score

    return (
        0.80 * bleu
        + 0.20 * chrf
    )


# =============================================================================
# Fingerprint inputs so a stale cache cannot be reused
# =============================================================================

references = (
    official_dev_df["reference_arabic"]
    .astype(str)
    .to_numpy(dtype=object)
)

fingerprint_hasher = hashlib.sha256()

for values in (
    baseline_predictions,
    oof_postedits,
    references
):
    for value in values:
        fingerprint_hasher.update(
            str(value).encode(
                "utf-8",
                errors="replace"
            )
        )

        fingerprint_hasher.update(b"\0")

pe9_input_fingerprint = (
    fingerprint_hasher.hexdigest()
)

pe9_cache_path = (
    PE_OUTPUT_DIR
    / "pe9_sentence_utilities_partial.npz"
)

baseline_utility = np.full(
    N,
    np.nan,
    dtype=np.float32
)

postedit_utility = np.full(
    N,
    np.nan,
    dtype=np.float32
)


# =============================================================================
# Load partial scoring cache if available
# =============================================================================

if pe9_cache_path.exists():
    try:
        with np.load(
            pe9_cache_path,
            allow_pickle=False
        ) as cache:
            cached_fingerprint = str(
                cache["input_fingerprint"].item()
            )

            cached_baseline = cache[
                "baseline_utility"
            ]

            cached_postedit = cache[
                "postedit_utility"
            ]

        cache_is_valid = (
            cached_fingerprint
            == pe9_input_fingerprint
            and cached_baseline.shape == (N,)
            and cached_postedit.shape == (N,)
        )

        if cache_is_valid:
            baseline_utility[:] = (
                cached_baseline
            )

            postedit_utility[:] = (
                cached_postedit
            )

            completed_rows = int(
                (
                    np.isfinite(
                        baseline_utility
                    )
                    & np.isfinite(
                        postedit_utility
                    )
                ).sum()
            )

            print(
                "Loaded PE-9 utility cache:",
                completed_rows,
                "/",
                N
            )

        else:
            print(
                "Ignoring stale PE-9 cache."
            )

    except Exception as error:
        print(
            "Ignoring unreadable PE-9 cache:",
            repr(error)
        )


def pe9_save_utility_cache():
    temporary_path = (
        pe9_cache_path.with_suffix(
            ".npz.tmp"
        )
    )

    with open(
        temporary_path,
        "wb"
    ) as file:
        np.savez_compressed(
            file,
            input_fingerprint=np.asarray(
                pe9_input_fingerprint
            ),
            baseline_utility=(
                baseline_utility
            ),
            postedit_utility=(
                postedit_utility
            )
        )

    os.replace(
        temporary_path,
        pe9_cache_path
    )


# =============================================================================
# Compute only missing sentence utilities
# =============================================================================

missing_utility_rows = np.where(
    ~np.isfinite(baseline_utility)
    | ~np.isfinite(postedit_utility)
)[0]

for completed_since_start, row_index in enumerate(
    tqdm(
        missing_utility_rows,
        desc="PE-9 sentence utilities"
    ),
    start=1
):
    reference = references[row_index]

    baseline_utility[row_index] = (
        pe_sentence_utility(
            baseline_predictions[
                row_index
            ],
            reference
        )
    )

    postedit_utility[row_index] = (
        pe_sentence_utility(
            oof_postedits[
                row_index
            ],
            reference
        )
    )

    if completed_since_start % 250 == 0:
        pe9_save_utility_cache()

pe9_save_utility_cache()

if not (
    np.isfinite(baseline_utility).all()
    and np.isfinite(postedit_utility).all()
):
    raise RuntimeError(
        "PE-9 sentence utilities are incomplete."
    )

gate_delta = (
    postedit_utility
    - baseline_utility
).astype(np.float32)


# =============================================================================
# Build gate features
# =============================================================================

def pe_similarity_normalized(
    first,
    second
):
    return SequenceMatcher(
        None,
        first,
        second
    ).ratio()


pe_dialects = sorted(
    official_dev_df["config"]
    .astype(str)
    .unique()
)

pe_dialect_to_id = {
    code: index
    for index, code in enumerate(
        pe_dialects
    )
}

number_of_numerical_features = 9

gate_features = np.zeros(
    (
        N,
        number_of_numerical_features
        + len(pe_dialects)
    ),
    dtype=np.float32
)

for row_index in tqdm(
    range(N),
    desc="PE-9 gate features"
):
    baseline = pe_norm(
        baseline_predictions[
            row_index
        ]
    )

    postedit = pe_norm(
        oof_postedits[
            row_index
        ]
    )

    alternative1 = pe_norm(
        candidate_texts[
            row_index,
            int(
                oof_alt1_idx[
                    row_index
                ]
            )
        ]
    )

    alternative2 = pe_norm(
        candidate_texts[
            row_index,
            int(
                oof_alt2_idx[
                    row_index
                ]
            )
        ]
    )

    baseline_length = max(
        1,
        len(baseline)
    )

    postedit_length = max(
        1,
        len(postedit)
    )

    gate_features[
        row_index,
        :number_of_numerical_features
    ] = [
        float(
            np.clip(
                oof_logprobs[
                    row_index
                ],
                -20,
                0
            )
        ),
        1.0 - pe_similarity_normalized(
            postedit,
            baseline
        ),
        math.log(
            postedit_length
            / baseline_length
        ),
        pe_similarity_normalized(
            postedit,
            alternative1
        ),
        pe_similarity_normalized(
            postedit,
            alternative2
        ),
        pe_similarity_normalized(
            alternative1,
            alternative2
        ),
        pe_similarity_normalized(
            baseline,
            alternative1
        ),
        pe_similarity_normalized(
            baseline,
            alternative2
        ),
        float(
            postedit == baseline
        )
    ]

    dialect_code = str(
        official_dev_df.iloc[
            row_index
        ]["config"]
    )

    dialect_position = (
        number_of_numerical_features
        + pe_dialect_to_id[
            dialect_code
        ]
    )

    gate_features[
        row_index,
        dialect_position
    ] = 1.0


# =============================================================================
# Create KEEP/EDIT labels
# =============================================================================

gate_labels = (
    gate_delta
    > GATE_LABEL_MIN_UTILITY_GAIN
).astype(np.int64)

np.save(
    PE_OUTPUT_DIR
    / "gate_features.npy",
    gate_features
)

np.save(
    PE_OUTPUT_DIR
    / "gate_delta.npy",
    gate_delta
)

np.save(
    PE_OUTPUT_DIR
    / "gate_labels.npy",
    gate_labels
)

print(
    "Actual OOF post-edit wins:",
    int(gate_labels.sum()),
    "/",
    N,
    f"({gate_labels.mean():.2%})"
)

print(
    "Mean sentence-utility delta:",
    float(gate_delta.mean())
)

print(
    "Gate feature matrix:",
    gate_features.shape
)

gc.collect()

PE-9 sentence utilities:   0%|          | 0/12250 [00:00<?, ?it/s]

PE-9 gate features:   0%|          | 0/12250 [00:00<?, ?it/s]

Actual OOF post-edit wins: 2541 / 12250 (20.74%)
Mean sentence-utility delta: -11.619078636169434
Gate feature matrix: (12250, 20)


35

In [17]:
def pe_official_metrics(
    predictions,
    indices,
    system_name
):
    indices = np.asarray(
        indices,
        dtype=np.int64
    )

    frame = official_dev_df.iloc[
        indices
    ][[
        "source_id",
        "config",
        "country",
        "conversation_id",
        "turn_order",
        "source_text",
        "reference_arabic"
    ]].copy()

    frame["prediction"] = np.asarray(
        predictions,
        dtype=object
    )

    rows = []

    for country in sorted(
        frame["config"].unique()
    ):
        country_df = frame[
            frame["config"] == country
        ]

        hypotheses = (
            country_df["prediction"]
            .astype(str)
            .tolist()
        )

        references = (
            country_df[
                "reference_arabic"
            ]
            .astype(str)
            .tolist()
        )

        rows.append({
            "country": country,
            "turns": len(country_df),
            "spBLEU": (
                sacrebleu.corpus_bleu(
                    hypotheses,
                    [references],
                    tokenize="flores200"
                ).score
            ),
            "chrF++": (
                sacrebleu.corpus_chrf(
                    hypotheses,
                    [references],
                    word_order=2
                ).score
            ),
            "BLEU": (
                sacrebleu.corpus_bleu(
                    hypotheses,
                    [references]
                ).score
            ),
            "chrF": (
                sacrebleu.corpus_chrf(
                    hypotheses,
                    [references],
                    word_order=0
                ).score
            )
        })

    per_country = pd.DataFrame(
        rows
    )

    summary = {
        "system": system_name,
        "Average spBLEU (primary)": float(
            per_country[
                "spBLEU"
            ].mean()
        ),
        "Average chrF++": float(
            per_country[
                "chrF++"
            ].mean()
        )
    }

    return (
        summary,
        per_country,
        frame
    )

def pe_calibrate_gate(
    probabilities,
    calibration_indices
):
    calibration_indices = np.asarray(
        calibration_indices,
        dtype=np.int64
    )

    baseline_summary, _, _ = (
        pe_official_metrics(
            baseline_predictions[
                calibration_indices
            ],
            calibration_indices,
            "baseline"
        )
    )

    baseline_score = float(
        baseline_summary[
            "Average spBLEU (primary)"
        ]
    )

    best_score = baseline_score
    best_threshold = 2.0
    best_count = 0

    calibration_probability = (
        probabilities[
            calibration_indices
        ]
    )

    thresholds = np.unique(np.r_[
        np.linspace(
            0.0,
            1.0,
            GATE_THRESHOLD_POINTS
        ),
        np.quantile(
            calibration_probability,
            np.linspace(
                0,
                1,
                GATE_THRESHOLD_POINTS
            )
        ),
        2.0
    ])

    for threshold in thresholds:
        accept = (
            calibration_probability
            >= threshold
        )

        accepted_count = int(
            accept.sum()
        )

        if (
            0
            < accepted_count
            < GATE_MIN_ACCEPTS
        ):
            continue

        predictions = np.where(
            accept,
            oof_postedits[
                calibration_indices
            ],
            baseline_predictions[
                calibration_indices
            ]
        )

        summary, _, _ = (
            pe_official_metrics(
                predictions,
                calibration_indices,
                "calibration"
            )
        )

        score = float(
            summary[
                "Average spBLEU (primary)"
            ]
        )

        if (
            score
            > best_score
            + 1e-12
        ):
            best_score = score
            best_threshold = float(
                threshold
            )
            best_count = (
                accepted_count
            )

    return (
        best_threshold,
        best_score - baseline_score,
        best_count
    )

gate_probability = np.zeros(
    N,
    dtype=np.float32
)

gate_threshold = np.full(
    N,
    2.0,
    dtype=np.float32
)

gate_metadata = []

for outer_fold in range(
    POSTEDIT_N_FOLDS
):
    outer_train = np.where(
        post_fold_id != outer_fold
    )[0]

    outer_heldout = np.where(
        post_fold_id == outer_fold
    )[0]

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=(
            GATE_CALIBRATION_FRACTION
        ),
        random_state=(
            PE_SEED
            + 5000
            + outer_fold
        )
    )

    (
        core_local,
        calibration_local
    ) = next(
        splitter.split(
            outer_train,
            groups=pe_groups[
                outer_train
            ]
        )
    )

    core_indices = outer_train[
        core_local
    ]

    calibration_indices = outer_train[
        calibration_local
    ]

    if len(np.unique(
        gate_labels[
            core_indices
        ]
    )) < 2:
        print(
            f"Gate fold {outer_fold + 1}: "
            "only one core label; "
            "keeping System92."
        )

        continue

    scaler = StandardScaler().fit(
        gate_features[
            core_indices
        ]
    )

    gate_model = LogisticRegression(
        C=0.5,
        class_weight="balanced",
        max_iter=2000,
        random_state=(
            PE_SEED
            + outer_fold
        )
    )

    gate_model.fit(
        scaler.transform(
            gate_features[
                core_indices
            ]
        ),
        gate_labels[
            core_indices
        ]
    )

    temporary_probability = np.zeros(
        N,
        dtype=np.float32
    )

    temporary_probability[
        calibration_indices
    ] = gate_model.predict_proba(
        scaler.transform(
            gate_features[
                calibration_indices
            ]
        )
    )[:, 1]

    (
        threshold,
        calibration_gain,
        calibration_accepts
    ) = pe_calibrate_gate(
        temporary_probability,
        calibration_indices
    )

    gate_probability[
        outer_heldout
    ] = gate_model.predict_proba(
        scaler.transform(
            gate_features[
                outer_heldout
            ]
        )
    )[:, 1]

    gate_threshold[
        outer_heldout
    ] = threshold

    gate_metadata.append({
        "outer_fold": outer_fold,
        "core_rows": len(
            core_indices
        ),
        "calibration_rows": len(
            calibration_indices
        ),
        "heldout_rows": len(
            outer_heldout
        ),
        "threshold": threshold,
        "calibration_spBLEU_gain": (
            calibration_gain
        ),
        "calibration_accepts": (
            calibration_accepts
        )
    })

    print(
        f"Gate fold {outer_fold + 1}: "
        f"threshold={threshold:.4f}, "
        f"calibration gain="
        f"{calibration_gain:+.4f}, "
        f"accepts={calibration_accepts}"
    )

accept_postedit = (
    gate_probability
    >= gate_threshold
)

gated_predictions = np.where(
    accept_postedit,
    oof_postedits,
    baseline_predictions
)

pd.DataFrame(
    gate_metadata
).to_csv(
    PE_OUTPUT_DIR
    / "gate_calibration_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Nested OOF accepted post-edits:",
    int(accept_postedit.sum()),
    "/",
    N
)

Gate fold 1: threshold=0.6750, calibration gain=+0.0207, accepts=11
Gate fold 2: threshold=0.7000, calibration gain=+0.0948, accepts=11
Nested OOF accepted post-edits: 163 / 12250


In [18]:
all_indices = np.arange(N)

(
    baseline_summary,
    baseline_per_country,
    baseline_scored
) = pe_official_metrics(
    baseline_predictions,
    all_indices,
    BASELINE_VARIANT
)

(
    raw_summary,
    raw_per_country,
    raw_scored
) = pe_official_metrics(
    oof_postedits,
    all_indices,
    RUN_NAME + "_raw_OOF"
)

(
    gated_summary,
    gated_per_country,
    gated_scored
) = pe_official_metrics(
    gated_predictions,
    all_indices,
    RUN_NAME + "_gated_OOF"
)

comparison = pd.DataFrame([
    baseline_summary,
    raw_summary,
    gated_summary
])

display(comparison)

country_comparison = (
    baseline_per_country[[
        "country",
        "turns",
        "spBLEU",
        "chrF++"
    ]]
    .rename(columns={
        "spBLEU": (
            "baseline_spBLEU"
        ),
        "chrF++": (
            "baseline_chrF++"
        )
    })
)

country_comparison = (
    country_comparison
    .merge(
        raw_per_country[[
            "country",
            "spBLEU",
            "chrF++"
        ]].rename(columns={
            "spBLEU": (
                "raw_spBLEU"
            ),
            "chrF++": (
                "raw_chrF++"
            )
        }),
        on="country"
    )
)

country_comparison = (
    country_comparison
    .merge(
        gated_per_country[[
            "country",
            "spBLEU",
            "chrF++"
        ]].rename(columns={
            "spBLEU": (
                "gated_spBLEU"
            ),
            "chrF++": (
                "gated_chrF++"
            )
        }),
        on="country"
    )
)

country_comparison[
    "raw_delta"
] = (
    country_comparison[
        "raw_spBLEU"
    ]
    - country_comparison[
        "baseline_spBLEU"
    ]
)

country_comparison[
    "gated_delta"
] = (
    country_comparison[
        "gated_spBLEU"
    ]
    - country_comparison[
        "baseline_spBLEU"
    ]
)

display(country_comparison)

system_candidates = [
    (
        BASELINE_VARIANT,
        baseline_predictions,
        baseline_summary,
        baseline_per_country,
        baseline_scored
    ),
    (
        RUN_NAME + "_raw_OOF",
        oof_postedits,
        raw_summary,
        raw_per_country,
        raw_scored
    ),
    (
        RUN_NAME + "_gated_OOF",
        gated_predictions,
        gated_summary,
        gated_per_country,
        gated_scored
    )
]

best_system = max(
    system_candidates,
    key=lambda item: (
        item[2][
            "Average spBLEU (primary)"
        ]
    )
)

baseline_score = baseline_summary[
    "Average spBLEU (primary)"
]

if (
    best_system[2][
        "Average spBLEU (primary)"
    ]
    <= baseline_score
    + DEPLOY_MIN_SPBLEU_GAIN
):
    best_system = (
        system_candidates[0]
    )

(
    final_system,
    final_predictions,
    final_summary,
    final_per_country,
    final_scored
) = best_system

print(
    "Raw OOF gain:",
    f"{raw_summary['Average spBLEU (primary)'] - baseline_score:+.6f}"
)

print(
    "Gated OOF gain:",
    f"{gated_summary['Average spBLEU (primary)'] - baseline_score:+.6f}"
)

print(
    "Deployment decision:",
    final_system
)

,system,Average spBLEU (primary),Average chrF++
0,92_mixed_best_checkpoint_variant_per_country,30.928003,45.594526
1,96_nilechat3b_multicandidate_posteditor_deadli...,18.066508,35.254299
2,96_nilechat3b_multicandidate_posteditor_deadli...,30.898240,45.578666


,country,turns,baseline_spBLEU,baseline_chrF++,raw_spBLEU,raw_chrF++,gated_spBLEU,gated_chrF++,raw_delta,gated_delta
0,EG,1113,32.903189,46.892198,20.220479,36.444722,32.845042,46.860362,-12.682710,-0.058146
1,JO,1113,35.301943,49.377895,21.431496,38.320206,35.367325,49.380479,-13.870446,0.065382
2,LB,1118,31.701457,45.871215,19.124524,35.942050,31.650208,45.832147,-12.576934,-0.051249
3,MA,1110,23.530143,39.376601,15.110064,31.867756,23.497257,39.370319,-8.420079,-0.032886
4,MR,1114,17.380438,33.733013,9.440650,27.106735,17.348151,33.750503,-7.939788,-0.032287
5,OM,1109,36.147408,49.843410,17.613881,35.505478,36.147408,49.843410,-18.533527,0.000000
6,PS,1110,33.418402,47.762506,18.838096,36.133914,33.433214,47.794268,-14.580305,0.014813
7,SA,1110,33.166094,48.135685,19.467287,37.461362,33.137598,48.124466,-13.698807,-0.028497
8,SY,1119,39.993111,53.985574,24.197206,42.028045,39.986454,53.996221,-15.795905,-0.006657
9,TN,1116,29.285225,43.452865,19.953573,35.635431,29.102676,43.324090,-9.331652,-0.182549


Raw OOF gain: -12.861494
Gated OOF gain: -0.029763
Deployment decision: 92_mixed_best_checkpoint_variant_per_country


In [19]:
turn_df = official_dev_df[[
    "source_id",
    "config",
    "country",
    "conversation_id",
    "turn_order",
    "source_text"
]].copy()

turn_df["prediction"] = np.asarray(
    final_predictions,
    dtype=object
)

turn_df[
    "system92_prediction"
] = baseline_predictions

turn_df[
    "raw_postedit"
] = oof_postedits

turn_df[
    "postedit_mean_logprob"
] = oof_logprobs

turn_df[
    "gate_probability"
] = gate_probability

turn_df[
    "gate_threshold"
] = gate_threshold

turn_df[
    "accepted_postedit"
] = accept_postedit

turn_df[
    "changed_from_system92"
] = [
    pe_norm(prediction)
    != pe_norm(baseline)
    for prediction, baseline
    in zip(
        turn_df["prediction"],
        baseline_predictions
    )
]

turn_df.to_csv(
    PE_OUTPUT_DIR
    / "turn_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

final_scored.to_csv(
    PE_OUTPUT_DIR
    / "scored_turn_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison.to_csv(
    PE_OUTPUT_DIR
    / "dev_system_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

country_comparison.to_csv(
    PE_OUTPUT_DIR
    / "dev_per_country_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

final_per_country.to_csv(
    PE_OUTPUT_DIR
    / "per_country_official_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

score_row = {
    "Variant": final_system,
    "Checkpoint": (
        "2-fold OOF post-editor"
    ),
    "Average spBLEU (primary)": (
        final_summary[
            "Average spBLEU (primary)"
        ]
    ),
    "Average chrF++": (
        final_summary[
            "Average chrF++"
        ]
    )
}

for row in final_per_country.to_dict(
    "records"
):
    score_row[
        f"{row['country']} spBLEU"
    ] = row["spBLEU"]

    score_row[
        f"{row['country']} chrF++"
    ] = row["chrF++"]

pd.DataFrame([
    score_row
]).to_csv(
    PE_OUTPUT_DIR
    / "official_leaderboard_score_row.csv",
    index=False,
    encoding="utf-8-sig"
)

pe_atomic_json_save({
    "run": RUN_NAME,
    "signature": PE_SIGNATURE,
    "deployed_system": final_system,
    "baseline": baseline_summary,
    "raw_posteditor": raw_summary,
    "gated_posteditor": gated_summary,
    "accepted_postedits": int(
        accept_postedit.sum()
    ),
    "sacrebleu_version": (
        sacrebleu.__version__
    )
}, (
    PE_OUTPUT_DIR
    / "official_metrics.json"
))

submission_records = []

ordered = final_scored.sort_values([
    "config",
    "conversation_id",
    "turn_order"
])

for (
    country,
    conversation_id
), conversation_df in ordered.groupby(
    [
        "config",
        "conversation_id"
    ],
    sort=True
):
    turns = [
        {
            "turn_order": int(
                row.turn_order
            ),
            "prediction": str(
                row.prediction
            )
        }
        for row
        in conversation_df.itertuples()
    ]

    submission_records.append({
        "conv_id": str(
            conversation_id
        ),
        "country": str(
            country
        ),
        "turns": turns
    })

jsonl_path = (
    PE_OUTPUT_DIR
    / "predictions.jsonl"
)

with open(
    jsonl_path,
    "w",
    encoding="utf-8"
) as file:
    for record in submission_records:
        file.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )

zip_path = (
    PE_OUTPUT_DIR
    / "submission_predictions.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    archive.write(
        jsonl_path,
        arcname="predictions.jsonl"
    )

readback_keys = set()
readback_turns = 0

with zipfile.ZipFile(
    zip_path
) as archive:
    if archive.namelist() != [
        "predictions.jsonl"
    ]:
        raise RuntimeError(
            "ZIP must contain only "
            "predictions.jsonl"
        )

    with archive.open(
        "predictions.jsonl"
    ) as file:
        for line in file:
            record = json.loads(
                line.decode("utf-8")
            )

            for turn in record["turns"]:
                readback_turns += 1

                readback_keys.add((
                    str(record["country"]),
                    str(record["conv_id"]),
                    int(turn["turn_order"])
                ))

expected_keys = set(zip(
    official_dev_df[
        "config"
    ].astype(str),
    official_dev_df[
        "conversation_id"
    ].astype(str),
    official_dev_df[
        "turn_order"
    ].astype(int)
))

if (
    readback_turns
    != EXPECTED_DEV_TURNS
    or readback_keys
    != expected_keys
):
    raise RuntimeError(
        "Submission ZIP readback "
        "validation failed."
    )

print(
    "\nOFFICIAL-STYLE "
    "DEVELOPMENT RESULT"
)

print(
    "System:",
    final_system
)

print(
    "Average spBLEU:",
    f"{final_summary['Average spBLEU (primary)']:.6f}"
)

print(
    "Average chrF++:",
    f"{final_summary['Average chrF++']:.6f}"
)

print(
    "Text-level changes from System92:",
    int(
        turn_df[
            "changed_from_system92"
        ].sum()
    ),
    "/",
    N
)

print(
    "\nSUBMIT THIS ZIP:",
    zip_path
)

display(
    final_per_country
)


OFFICIAL-STYLE DEVELOPMENT RESULT
System: 92_mixed_best_checkpoint_variant_per_country
Average spBLEU: 30.928003
Average chrF++: 45.594526
Text-level changes from System92: 0 / 12250

SUBMIT THIS ZIP: /home/mabdallah/alexandriax_mt_14d/inference_variants/96_nilechat3b_multicandidate_posteditor_deadline_v1/submission_predictions.zip


,country,turns,spBLEU,chrF++,BLEU,chrF
0,EG,1113,32.903189,46.892198,19.510451,49.894903
1,JO,1113,35.301943,49.377895,19.819243,52.901916
2,LB,1118,31.701457,45.871215,20.021014,48.586333
3,MA,1110,23.530143,39.376601,12.865149,42.236961
4,MR,1114,17.380438,33.733013,7.100699,37.779153
5,OM,1109,36.147408,49.843410,20.805106,53.247687
6,PS,1110,33.418402,47.762506,20.699325,51.031341
7,SA,1110,33.166094,48.135685,18.903768,52.048449
8,SY,1119,39.993111,53.985574,26.598272,56.940070
9,TN,1116,29.285225,43.452865,17.062352,46.059624
